# BDR PDF screening, OCR and CGWB parameters

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Screens the Alwar BDR reports (5 PDFs, 1,008 pages) and the GEC 2024-25 workbook for lithology, boreholes and pumping tests, with OCR and a visual-review packet. **Status: nothing is verified yet.** The readiness audit flags hydraulic parameters, georeferenced boreholes and lithological intervals as *not verified*. The CGWB report download failed because of an SSL error.

| | |
|---|---|
| **Reads** | `gw_framework/data/BDR/`, `gw_framework/data/CGRA_Reports/stateReport 2024-25.xlsx` |
| **Writes** | `MyDrive/Groundwater_BDR_Analysis_Output/`, `MyDrive/Alwar_Verified_Data_Readiness_Audit/`, `MyDrive/Alwar_BDR_Lithology_Inventory/` (these folders sit outside `gw_framework`; see the README for where to move them) |
| **Cells** | 16 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Scan BDR reports and readiness audit
- 2. OCR and visual review
- 3. CGWB hydraulic parameters

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["pdfplumber", "pymupdf", "pytesseract", "openpyxl"], modflow=False)
del TOKEN, URL


## 1. Scan BDR reports and readiness audit

**[orig #13]** Scan the BDR PDFs (Alwar ×5, Bharatpur ×1) for lithology / borehole / pumping-test keywords  
<sub>✅ ran OK</sub>

In [ ]:
# =====================================================================
# BDR REPORT ANALYSIS FOR GROUNDWATER MODELLING
# Districts: Alwar, Dholpur, Bharatpur + any other districts discovered
#
# Outputs:
# 1. PDF inventory and extraction quality
# 2. Page-by-page extracted text
# 3. Hydrogeological keyword evidence by page
# 4. Candidate borehole / pumping-test / aquifer evidence
# 5. District-wise data-readiness screening
# 6. PDF processing errors and OCR report
#
# Original PDFs are never modified.
# =====================================================================

# -------------------------- 1. INSTALL -------------------------------

import sys
import subprocess
import importlib.util

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

install_if_missing("PyMuPDF", "fitz")
install_if_missing("pandas")
install_if_missing("pytesseract")
install_if_missing("Pillow", "PIL")
install_if_missing("openpyxl")

# Install Tesseract OCR if unavailable.
if subprocess.run(
    ["bash", "-lc", "command -v tesseract"],
    capture_output=True, text=True
).returncode != 0:
    subprocess.run(
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq tesseract-ocr"],
        check=True
    )

# -------------------------- 2. IMPORTS -------------------------------

from pathlib import Path
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np
import fitz
import pytesseract
import re
import json
import os
import gc
import traceback

from PIL import Image

# -------------------------- 3. GOOGLE DRIVE -------------------------

from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

# Screenshot shows: data > BDR
PREFERRED_BDR = MYDRIVE / "data" / "BDR"

# You can replace the line above with your exact folder path if needed.
# Example:
# PREFERRED_BDR = MYDRIVE / "some_folder" / "BDR"

# -------------------------- 4. FIND BDR FOLDER -----------------------

def find_bdr_folders(root):
    found = []
    for current, dirs, files in os.walk(root):
        # Avoid searching trash/system folders.
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".Trash", ".trash", ".ipynb_checkpoints",
                "Groundwater_BDR_Analysis_Output"
            }
        ]
        if Path(current).name.casefold() == "bdr":
            found.append(Path(current))
    return found

if PREFERRED_BDR.is_dir():
    BDR_DIR = PREFERRED_BDR
else:
    print("Preferred path not found:", PREFERRED_BDR)
    candidates = find_bdr_folders(MYDRIVE)

    if len(candidates) == 1:
        BDR_DIR = candidates[0]
    elif len(candidates) == 0:
        raise FileNotFoundError(
            "No folder named BDR was found in MyDrive.\n"
            "Set PREFERRED_BDR to the correct Google Drive folder path."
        )
    else:
        print("Multiple BDR folders found:")
        for i, p in enumerate(candidates, 1):
            print(f"{i}. {p}")
        raise RuntimeError(
            "Multiple BDR folders found. Set PREFERRED_BDR manually "
            "to the folder shown in your Google Drive screenshot."
        )

print("\nBDR folder selected:")
print(BDR_DIR)

# Save results outside the source BDR folder.
OUTPUT_DIR = MYDRIVE / "Groundwater_BDR_Analysis_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_DIR = OUTPUT_DIR / "Extracted_Text"
TEXT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------- 5. PDF DISCOVERY -------------------------

pdf_files = sorted(
    [p for p in BDR_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"]
)

# Avoid accidentally processing prior output files if the output folder
# happens to be nested under a searched folder.
pdf_files = [
    p for p in pdf_files
    if OUTPUT_DIR not in p.parents
]

print(f"\nPDF files discovered: {len(pdf_files)}")

if not pdf_files:
    raise RuntimeError(
        "No PDFs found. Check that BDR_DIR points to the folder "
        "containing the actual reports."
    )

for p in pdf_files:
    print(f" - {p.relative_to(BDR_DIR)}")

# -------------------------- 6. ANALYSIS TERMS ------------------------

# These are screening terms, not proof that a report contains usable data.
KEYWORDS = {
    "lithology_and_logs": [
        "litholog", "lithological log", "bore log", "borehole log",
        "drill log", "geological log", "litho log", "stratigraphy",
        "geological formation", "formation encountered"
    ],
    "boreholes_and_wells": [
        "borehole", "bore hole", "exploratory well", "exploratory bore",
        "tube well", "tubewell", "observation well", "piezometer",
        "well inventory", "well depth", "drilled depth", "total depth"
    ],
    "coordinates_and_location": [
        "latitude", "longitude", "coordinates", "gps", "easting",
        "northing", "toposheet", "location map", "site location"
    ],
    "aquifer_geometry": [
        "aquifer", "aquifer thickness", "saturated thickness",
        "water-bearing zone", "water bearing zone", "fractured zone",
        "weathered zone", "alluvium", "sandstone", "limestone",
        "granite", "gneiss", "schist", "quartzite", "clay", "gravel"
    ],
    "pumping_tests": [
        "pumping test", "pumping-test", "aquifer test",
        "step drawdown", "step-drawdown", "constant discharge",
        "constant rate test", "recovery test", "drawdown",
        "discharge", "specific capacity"
    ],
    "hydraulic_parameters": [
        "hydraulic conductivity", "permeability", "transmissivity",
        "transmissivity value", "storativity", "storage coefficient",
        "specific yield", "hydraulic diffusivity"
    ],
    "groundwater_monitoring": [
        "ground water level", "groundwater level", "water level",
        "water table", "depth to water", "pre monsoon", "post monsoon",
        "hydrograph", "monitoring station", "observation network"
    ],
    "recharge_and_stress": [
        "groundwater recharge", "recharge", "replenishable resource",
        "extractable resource", "stage of extraction", "groundwater draft",
        "annual extraction", "irrigation draft", "rainfall",
        "water balance", "groundwater budget"
    ],
    "model_boundary_and_geophysics": [
        "electrical resistivity", "geophysical survey",
        "vertical electrical sounding", "ves", "contour map",
        "groundwater flow", "hydraulic gradient", "drainage",
        "river boundary", "impermeable boundary", "aquifer boundary"
    ],
    "model_ready_data": [
        "layer thickness", "top elevation", "bottom elevation",
        "recharge rate", "hydraulic head", "observation date",
        "water level measurement", "pumping rate"
    ]
}

# -------------------------- 7. HELPER FUNCTIONS ----------------------

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def safe_filename(text, max_length=150):
    text = re.sub(r'[<>:"/\\|?*]+', "_", str(text))
    text = re.sub(r"\s+", "_", text).strip("_")
    return text[:max_length] or "unnamed"

def infer_district(text, file_path):
    """
    Infer district from filename/path first, then document text.
    The result is a candidate label and must be reviewed.
    """
    source = (str(file_path) + " " + text[:12000]).casefold()

    district_names = [
        "Alwar", "Dholpur", "Bharatpur", "Jaipur", "Dausa",
        "Karauli", "Sawai Madhopur", "Tonk", "Kota", "Baran",
        "Jhalawar", "Sikar", "Jhunjhunu", "Ajmer", "Bikaner",
        "Jodhpur", "Udaipur", "Banswara", "Bhilwara", "Churu",
        "Chittorgarh", "Dungarpur", "Hanumangarh", "Jaisalmer",
        "Jalore", "Pali", "Rajsamand", "Sirohi", "Sri Ganganagar",
        "Barmer", "Nagaur", "Pratapgarh", "Kekri", "Kotputli-Behror",
        "Khairthal-Tijara", "Deeg", "Didwana-Kuchaman", "Phalodi",
        "Salumbar", "Gangapur City", "Balotra"
    ]

    hits = []
    for name in district_names:
        pattern = r"(?<![a-z])" + re.escape(name.casefold()) + r"(?![a-z])"
        if re.search(pattern, source):
            hits.append(name)

    if len(hits) == 1:
        return hits[0], "single district name detected"
    elif len(hits) > 1:
        # If the file path or filename identifies one district, prefer it.
        path_text = str(file_path).casefold()
        path_hits = [
            n for n in hits
            if re.search(
                r"(?<![a-z])" + re.escape(n.casefold()) + r"(?![a-z])",
                path_text
            )
        ]
        if len(path_hits) == 1:
            return path_hits[0], "district inferred from file path/name"
        return "Multiple/uncertain", "; ".join(hits)

    return "Unknown", "no district name detected"

def count_keyword_matches(text, keyword_list):
    low = text.casefold()
    return sum(low.count(k.casefold()) for k in keyword_list)

def matching_terms(text, keyword_list):
    low = text.casefold()
    return "; ".join(
        k for k in keyword_list if k.casefold() in low
    )

def candidate_numeric_evidence(text):
    """
    Extract candidate strings only. These are not validated measurements.
    """
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    patterns = {
        "coordinate_like_lines": re.compile(
            r"\b(?:latitude|longitude|lat\.?|long\.?|easting|northing|"
            r"coordinates|coord\.?)\b", re.I
        ),
        "depth_like_lines": re.compile(
            r"\b(?:depth|borehole|drilled|well depth|thickness)\b", re.I
        ),
        "hydraulic_parameter_lines": re.compile(
            r"\b(?:transmissivity|hydraulic conductivity|permeability|"
            r"storativity|storage coefficient|specific yield)\b", re.I
        ),
        "pumping_test_lines": re.compile(
            r"\b(?:pumping test|drawdown|recovery test|discharge|"
            r"specific capacity)\b", re.I
        )
    }

    found = {}
    for label, pattern in patterns.items():
        found[label] = [
            line[:500] for line in lines if pattern.search(line)
        ][:100]
    return found

# -------------------------- 8. PDF EXTRACTION ------------------------

document_rows = []
page_rows = []
keyword_rows = []
error_rows = []
candidate_rows = []

for index, pdf_path in enumerate(pdf_files, 1):
    print(
        f"\n[{index}/{len(pdf_files)}] Processing: "
        f"{pdf_path.relative_to(BDR_DIR)}"
    )

    try:
        file_size_mb = pdf_path.stat().st_size / (1024 * 1024)
        doc = fitz.open(pdf_path)

        page_texts = []
        page_ocr_flags = []
        pages_with_text = 0
        pages_ocr_attempted = 0
        pages_ocr_used = 0
        ocr_errors = 0

        for page_index in range(len(doc)):
            page = doc[page_index]
            text = clean_text(page.get_text("text"))
            ocr_used = False
            ocr_attempted = False
            ocr_error = ""

            # OCR pages with little/no extractable text.
            # The threshold can be adjusted if needed.
            if len(text) < 50:
                ocr_attempted = True
                pages_ocr_attempted += 1

                try:
                    pix = page.get_pixmap(
                        matrix=fitz.Matrix(2, 2),
                        colorspace=fitz.csRGB,
                        alpha=False
                    )
                    image = Image.frombytes(
                        "RGB", [pix.width, pix.height], pix.samples
                    )
                    ocr_text = clean_text(
                        pytesseract.image_to_string(image, lang="eng")
                    )

                    # Use OCR if it recovers more text.
                    if len(ocr_text) > len(text):
                        text = ocr_text
                        ocr_used = True
                        pages_ocr_used += 1

                    del image, pix
                    gc.collect()

                except Exception as e:
                    ocr_errors += 1
                    ocr_error = str(e)[:500]

            if len(text) > 0:
                pages_with_text += 1

            page_texts.append(text)
            page_ocr_flags.append(ocr_used)

            district, district_reason = infer_district(text, pdf_path)

            page_record = {
                "pdf_file": pdf_path.name,
                "relative_path": str(pdf_path.relative_to(BDR_DIR)),
                "page_number": page_index + 1,
                "district_candidate": district,
                "district_inference_note": district_reason,
                "text_characters": len(text),
                "ocr_attempted": ocr_attempted,
                "ocr_used": ocr_used,
                "ocr_error": ocr_error,
                "page_text": text
            }
            page_rows.append(page_record)

            page_keyword_record = {
                "pdf_file": pdf_path.name,
                "relative_path": str(pdf_path.relative_to(BDR_DIR)),
                "page_number": page_index + 1,
                "district_candidate": district
            }

            for category, terms in KEYWORDS.items():
                page_keyword_record[category + "_count"] = (
                    count_keyword_matches(text, terms)
                )
                page_keyword_record[category + "_terms"] = (
                    matching_terms(text, terms)
                )

            keyword_rows.append(page_keyword_record)

            candidates = candidate_numeric_evidence(text)
            for category, lines in candidates.items():
                for line in lines:
                    candidate_rows.append({
                        "pdf_file": pdf_path.name,
                        "relative_path": str(pdf_path.relative_to(BDR_DIR)),
                        "page_number": page_index + 1,
                        "district_candidate": district,
                        "evidence_category": category,
                        "candidate_text": line
                    })

        full_text = "\n\n".join(
            f"===== PAGE {i + 1} =====\n{t}"
            for i, t in enumerate(page_texts)
        )

        district, district_reason = infer_district(full_text, pdf_path)

        # Save full extracted text, including OCR-recovered pages.
        text_filename = safe_filename(
            str(pdf_path.relative_to(BDR_DIR))
        ) + ".txt"

        (TEXT_DIR / text_filename).write_text(
            full_text, encoding="utf-8"
        )

        keyword_totals = {}
        for category, terms in KEYWORDS.items():
            keyword_totals[category] = count_keyword_matches(
                full_text, terms
            )

        document_rows.append({
            "pdf_file": pdf_path.name,
            "relative_path": str(pdf_path.relative_to(BDR_DIR)),
            "file_size_MB": round(file_size_mb, 3),
            "page_count": len(doc),
            "pages_with_extracted_text": pages_with_text,
            "pages_without_extracted_text": len(doc) - pages_with_text,
            "pages_ocr_attempted": pages_ocr_attempted,
            "pages_ocr_used": pages_ocr_used,
            "ocr_errors": ocr_errors,
            "total_extracted_characters": len(full_text),
            "district_candidate": district,
            "district_inference_note": district_reason,
            **keyword_totals,
            "extracted_text_file": str(TEXT_DIR / text_filename),
            "processing_status": "SUCCESS"
        })

        doc.close()

    except Exception as e:
        error_rows.append({
            "pdf_file": pdf_path.name,
            "relative_path": str(pdf_path),
            "error": str(e),
            "traceback": traceback.format_exc()[-3000:]
        })

        document_rows.append({
            "pdf_file": pdf_path.name,
            "relative_path": str(pdf_path),
            "file_size_MB": round(pdf_path.stat().st_size / (1024 * 1024), 3),
            "page_count": np.nan,
            "pages_with_extracted_text": np.nan,
            "pages_without_extracted_text": np.nan,
            "pages_ocr_attempted": np.nan,
            "pages_ocr_used": np.nan,
            "ocr_errors": np.nan,
            "total_extracted_characters": 0,
            "district_candidate": "Unknown",
            "district_inference_note": "PDF processing failed",
            **{k: 0 for k in KEYWORDS},
            "extracted_text_file": "",
            "processing_status": "FAILED"
        })

# -------------------------- 9. SAVE EXTRACTION REPORTS --------------

documents_df = pd.DataFrame(document_rows)
pages_df = pd.DataFrame(page_rows)
keywords_df = pd.DataFrame(keyword_rows)
candidates_df = pd.DataFrame(candidate_rows)
errors_df = pd.DataFrame(error_rows)

documents_df.to_csv(
    OUTPUT_DIR / "01_pdf_inventory_and_extraction_quality.csv",
    index=False
)
pages_df.to_csv(
    OUTPUT_DIR / "02_page_level_extracted_text.csv",
    index=False
)
keywords_df.to_csv(
    OUTPUT_DIR / "03_page_level_hydrogeology_keyword_evidence.csv",
    index=False
)
candidates_df.to_csv(
    OUTPUT_DIR / "04_candidate_numeric_and_parameter_evidence.csv",
    index=False
)
errors_df.to_csv(
    OUTPUT_DIR / "05_pdf_processing_errors.csv",
    index=False
)

# -------------------------- 10. DISTRICT SUMMARY --------------------

if not documents_df.empty:
    count_cols = list(KEYWORDS.keys())

    district_summary = (
        documents_df.groupby("district_candidate", dropna=False)
        .agg(
            pdf_count=("pdf_file", "count"),
            successful_pdfs=(
                "processing_status",
                lambda s: (s == "SUCCESS").sum()
            ),
            total_pages=("page_count", "sum"),
            total_extracted_characters=(
                "total_extracted_characters", "sum"
            ),
            pdfs_with_lithology_keywords=(
                "lithology_and_logs",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pdfs_with_borehole_keywords=(
                "boreholes_and_wells",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pdfs_with_pumping_test_keywords=(
                "pumping_tests",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pdfs_with_hydraulic_parameter_keywords=(
                "hydraulic_parameters",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pdfs_with_monitoring_keywords=(
                "groundwater_monitoring",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pdfs_with_recharge_keywords=(
                "recharge_and_stress",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            )
        )
        .reset_index()
    )

    district_summary.to_csv(
        OUTPUT_DIR / "06_district_wise_report_screening.csv",
        index=False
    )
else:
    district_summary = pd.DataFrame()

# -------------------------- 11. PRIORITY DISTRICT COMPARISON --------

priority_districts = ["Alwar", "Dholpur", "Bharatpur"]

priority_df = documents_df[
    documents_df["district_candidate"].isin(priority_districts)
].copy()

if not priority_df.empty:
    priority_summary = (
        priority_df.groupby("district_candidate")
        .agg(
            report_count=("pdf_file", "nunique"),
            successful_reports=(
                "processing_status",
                lambda s: (s == "SUCCESS").sum()
            ),
            total_pages=("page_count", "sum"),
            total_text_characters=("total_extracted_characters", "sum"),
            lithology_keyword_reports=(
                "lithology_and_logs",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            borehole_keyword_reports=(
                "boreholes_and_wells",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            pumping_test_keyword_reports=(
                "pumping_tests",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            hydraulic_parameter_keyword_reports=(
                "hydraulic_parameters",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            monitoring_keyword_reports=(
                "groundwater_monitoring",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            ),
            recharge_keyword_reports=(
                "recharge_and_stress",
                lambda s: (pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum()
            )
        )
        .reset_index()
    )

    # Explicitly avoid converting keyword hits into confirmed data counts.
    priority_summary["interpretation_note"] = (
        "Screening only: keyword presence does not confirm usable data."
    )

else:
    priority_summary = pd.DataFrame({
        "note": [
            "No reports were confidently assigned to Alwar, Dholpur, or Bharatpur. "
            "Review the PDF inventory and district inference notes."
        ]
    })

priority_summary.to_csv(
    OUTPUT_DIR / "07_priority_district_comparison.csv",
    index=False
)

# -------------------------- 12. PAGE-LEVEL DISTRICT EVIDENCE --------

if not keywords_df.empty:
    priority_pages = keywords_df[
        keywords_df["district_candidate"].isin(priority_districts)
    ].copy()

    priority_pages.to_csv(
        OUTPUT_DIR / "08_priority_district_page_evidence.csv",
        index=False
    )

# -------------------------- 13. DUPLICATE / OVERLAP SCREENING -------

# This is a filename-level screening, not a guarantee that two reports
# are duplicates. Different reports can legitimately repeat tables.
if not documents_df.empty:
    documents_df["filename_normalized"] = (
        documents_df["pdf_file"]
        .str.casefold()
        .str.replace(r"[^a-z0-9]", "", regex=True)
    )

    duplicate_name_screen = documents_df[
        documents_df.duplicated("filename_normalized", keep=False)
    ].copy()

    duplicate_name_screen.to_csv(
        OUTPUT_DIR / "09_possible_duplicate_filenames.csv",
        index=False
    )

# -------------------------- 14. RUN SUMMARY -------------------------

run_summary = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "source_folder": str(BDR_DIR),
    "output_folder": str(OUTPUT_DIR),
    "pdf_files_discovered": len(pdf_files),
    "pdfs_processed_successfully": int(
        (documents_df["processing_status"] == "SUCCESS").sum()
    ) if not documents_df.empty else 0,
    "pdfs_failed": int(
        (documents_df["processing_status"] == "FAILED").sum()
    ) if not documents_df.empty else 0,
    "total_pages_processed": int(
        pd.to_numeric(documents_df["page_count"], errors="coerce").sum()
    ) if not documents_df.empty else 0,
    "pages_where_ocr_was_used": int(
        pd.to_numeric(documents_df["pages_ocr_used"], errors="coerce").sum()
    ) if not documents_df.empty else 0,
    "priority_districts": priority_districts,
    "important_limitation":
        "Automated keyword and OCR screening does not validate individual "
        "borehole logs, parameter units, coordinates, or data usability."
}

with open(OUTPUT_DIR / "00_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

# -------------------------- 15. DISPLAY RESULTS ---------------------

print("\n" + "=" * 78)
print("BDR REPORT SCANNING COMPLETED")
print("=" * 78)
print(json.dumps(run_summary, indent=2))

print("\nPDF INVENTORY")
display(
    documents_df[
        [
            "pdf_file", "page_count", "pages_with_extracted_text",
            "pages_ocr_used", "district_candidate",
            "lithology_and_logs", "boreholes_and_wells",
            "pumping_tests", "hydraulic_parameters",
            "groundwater_monitoring", "processing_status"
        ]
    ].head(100)
)

print("\nALWAR / DHOLPUR / BHARATPUR SCREENING")
display(priority_summary)

print("\nDISTRICT-WISE SCREENING")
display(district_summary)

print("\nOUTPUT FOLDER:")
print(OUTPUT_DIR)

print("\nIMPORTANT:")
print(
    "Use the keyword reports to locate promising pages. "
    "Before selecting districts, verify actual borehole coordinates, "
    "logged depth intervals, pumping-test values, parameter units, "
    "and monitoring coverage from the extracted text and original PDFs."
)

**[orig #14]** Alwar data-readiness audit (GEC 2024-25 workbook + BDR evidence)  
<sub>✅ ran OK</sub>

In [ ]:
# =====================================================================
# ALWAR GROUNDWATER DATA-READINESS AUDIT
# Sources:
#   1. BDR PDF extraction outputs
#   2. State groundwater resource assessment 2024-25 Excel workbook
#
# Does not modify original PDFs, Excel files, or existing extraction files.
# =====================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

# -------------------------- 1. PATHS ---------------------------------

MYDRIVE = Path("/content/drive/MyDrive")

BDR_DIR = MYDRIVE / "gw_framework/data/BDR"
PDF_AUDIT_DIR = MYDRIVE / "Groundwater_BDR_Analysis_Output"
EXCEL_FILE = MYDRIVE / "data/CGRA_Reports/stateReport 2024-25.xlsx"

# If the workbook is elsewhere, locate it by filename.
if not EXCEL_FILE.exists():
    matches = list(MYDRIVE.rglob("stateReport 2024-25.xlsx"))
    if len(matches) == 1:
        EXCEL_FILE = matches[0]
    elif len(matches) == 0:
        raise FileNotFoundError(
            "Could not find stateReport 2024-25.xlsx in MyDrive. "
            "Update EXCEL_FILE to its actual path."
        )
    else:
        raise FileNotFoundError(
            "Multiple files named stateReport 2024-25.xlsx were found. "
            "Set EXCEL_FILE to the correct one."
        )

PDF_INVENTORY = (
    PDF_AUDIT_DIR / "01_pdf_inventory_and_extraction_quality.csv"
)
PDF_PAGE_EVIDENCE = (
    PDF_AUDIT_DIR / "03_page_level_hydrogeology_keyword_evidence.csv"
)
PDF_CANDIDATE_LINES = (
    PDF_AUDIT_DIR / "04_candidate_numeric_and_parameter_evidence.csv"
)
PDF_PAGE_TEXT = (
    PDF_AUDIT_DIR / "02_page_level_extracted_text.csv"
)

OUT = MYDRIVE / "Alwar_Verified_Data_Readiness_Audit"
OUT.mkdir(parents=True, exist_ok=True)

print("Excel workbook:", EXCEL_FILE)
print("BDR folder:", BDR_DIR)
print("PDF extraction folder:", PDF_AUDIT_DIR)
print("Audit output folder:", OUT)

# -------------------------- 2. LOAD PDF INVENTORY --------------------

if not PDF_INVENTORY.exists():
    raise FileNotFoundError(
        f"Missing {PDF_INVENTORY}. Run the BDR PDF scanning script first."
    )

pdfs = pd.read_csv(PDF_INVENTORY, low_memory=False)
pdfs.columns = pdfs.columns.str.strip()

if "district_candidate" not in pdfs.columns:
    raise KeyError("PDF inventory lacks district_candidate.")

alwar_pdfs = pdfs[
    pdfs["district_candidate"].astype(str).str.strip().str.casefold()
    == "alwar"
].copy()

# Preserve screening counts, but explicitly label them as keyword evidence.
screen_cols = [
    "pdf_file", "relative_path", "page_count",
    "pages_with_extracted_text", "pages_ocr_used",
    "total_extracted_characters", "processing_status",
    "lithology_and_logs", "boreholes_and_wells",
    "coordinates_and_location", "aquifer_geometry",
    "pumping_tests", "hydraulic_parameters",
    "groundwater_monitoring", "recharge_and_stress",
    "model_boundary_and_geophysics", "model_ready_data"
]
screen_cols = [c for c in screen_cols if c in alwar_pdfs.columns]
alwar_pdfs[screen_cols].to_csv(
    OUT / "01_Alwar_BDR_report_inventory.csv", index=False
)

print("\nAlwar PDFs in prior scan:", len(alwar_pdfs))
if not alwar_pdfs.empty:
    print("Total pages:", pd.to_numeric(
        alwar_pdfs["page_count"], errors="coerce"
    ).sum())

# -------------------------- 3. LOAD PAGE EVIDENCE --------------------

if PDF_PAGE_EVIDENCE.exists():
    page_evidence = pd.read_csv(
        PDF_PAGE_EVIDENCE, low_memory=False
    )
    page_evidence.columns = page_evidence.columns.str.strip()

    if "district_candidate" in page_evidence.columns:
        page_evidence = page_evidence[
            page_evidence["district_candidate"]
            .astype(str).str.strip().str.casefold().eq("alwar")
        ].copy()

    page_evidence.to_csv(
        OUT / "02_Alwar_page_level_BDR_evidence.csv", index=False
    )
else:
    page_evidence = pd.DataFrame()
    print("Warning: page-level PDF evidence file not found.")

# -------------------------- 4. CANDIDATE PARAMETER LINES --------------

if PDF_CANDIDATE_LINES.exists():
    candidate_lines = pd.read_csv(
        PDF_CANDIDATE_LINES, low_memory=False
    )
    candidate_lines.columns = candidate_lines.columns.str.strip()

    if "district_candidate" in candidate_lines.columns:
        candidate_lines = candidate_lines[
            candidate_lines["district_candidate"]
            .astype(str).str.strip().str.casefold().eq("alwar")
        ].copy()

    candidate_lines.to_csv(
        OUT / "03_Alwar_candidate_parameter_lines.csv", index=False
    )
else:
    candidate_lines = pd.DataFrame()
    print("Warning: candidate parameter lines file not found.")

# -------------------------- 5. EXTRACT ALWAR RESOURCE ASSESSMENT ------

# The GEC sheet uses multi-row headers. Read without assuming one header.
raw = pd.read_excel(
    EXCEL_FILE,
    sheet_name="GEC",
    header=None,
    dtype=object
)

# Based on the workbook layout, rows 7-9 (zero-based) contain the
# multi-level header and data begin after those rows.
# We build a label from all non-empty header cells in each column.
HEADER_ROWS = [7, 8, 9]
DATA_START_ROW = 10

def is_blank(v):
    return pd.isna(v) or str(v).strip() == ""

def header_piece(v):
    if is_blank(v):
        return ""
    return re.sub(r"\s+", " ", str(v).strip())

labels = []
for j in range(raw.shape[1]):
    pieces = []
    for r in HEADER_ROWS:
        piece = header_piece(raw.iat[r, j])
        if piece and piece not in pieces:
            pieces.append(piece)

    labels.append(" | ".join(pieces) if pieces else f"UNNAMED_COLUMN_{j+1}")

# Keep the original Excel column position in every output label.
labels = [f"COL_{j+1:03d} | {name}" for j, name in enumerate(labels)]

resource = raw.iloc[DATA_START_ROW:].copy()
resource.columns = labels
resource = resource.dropna(how="all").copy()

# Find district and block columns using the header labels.
def find_header_col(columns, phrase):
    matches = [c for c in columns if phrase in c.casefold()]
    return matches[0] if matches else None

district_col = find_header_col(resource.columns, "district")
block_col = find_header_col(resource.columns, "block")
state_col = find_header_col(resource.columns, "state")
serial_col = find_header_col(resource.columns, "s.no")

if district_col is None or block_col is None:
    raise KeyError(
        "Could not identify DISTRICT and BLOCK columns from the multi-row "
        f"headers. Found district={district_col}, block={block_col}. "
        "Inspect the workbook header rows before proceeding."
    )

resource["_district_normalized"] = (
    resource[district_col].astype(str).str.strip().str.upper()
)
resource["_block_original"] = resource[block_col].astype(str).str.strip()

alwar = resource[
    resource["_district_normalized"].eq("ALWAR")
].copy()

# Retain a transparent extract with original multi-level header names.
alwar.to_csv(
    OUT / "04_Alwar_GEC_2024_25_raw_column_labels.csv",
    index=False
)

# -------------------------- 6. BLOCK-LEVEL DUPLICATE CHECK ------------

alwar["_block_key"] = (
    alwar["_block_original"]
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

valid_blocks = alwar[
    alwar["_block_key"].ne("")
    & ~alwar["_block_key"].isin(["NAN", "NONE"])
].copy()

duplicate_block_mask = valid_blocks.duplicated(
    "_block_key", keep=False
)

duplicate_blocks = valid_blocks[duplicate_block_mask].copy()
duplicate_blocks.to_csv(
    OUT / "05_Alwar_duplicate_block_rows_review.csv",
    index=False
)

block_counts = (
    valid_blocks.groupby("_block_key", dropna=False)
    .size()
    .rename("rows_for_normalized_block")
    .reset_index()
)

block_counts.to_csv(
    OUT / "06_Alwar_block_row_counts.csv", index=False
)

# Do not automatically deduplicate assessment rows. Multiple rows may
# reflect category splits (C / NC / PQ) or multiple assessment records.

# -------------------------- 7. MAP IMPORTANT RESOURCE FIELDS ----------

# Find columns from their multi-level labels. The source labels are retained.
field_patterns = {
    "annual_groundwater_recharge": [
        "annual ground water recharge"
    ],
    "environmental_flows": [
        "environmental flows"
    ],
    "annual_extractable_resource": [
        "annual extractable ground water resource"
    ],
    "domestic_extraction": [
        "domestic"
    ],
    "industrial_extraction": [
        "industrial"
    ],
    "irrigation_extraction": [
        "irrigation"
    ],
    "stage_of_extraction": [
        "stage of ground water extraction"
    ],
    "assessment_category": [
        "categorization of assessment unit"
    ],
    "pre_monsoon_trend": [
        "pre monsoon of gw trend"
    ],
    "post_monsoon_trend": [
        "post monsoon of gw trend"
    ],
    "future_use_resource": [
        "net annual ground water availability for future use"
    ],
    "domestic_allocation_2025": [
        "allocation of ground water resource for domestic utilisation"
    ]
}

field_map = {}
for friendly_name, patterns in field_patterns.items():
    matches = []
    for col in alwar.columns:
        if col.startswith("_"):
            continue
        col_low = col.casefold()
        if any(p in col_low for p in patterns):
            matches.append(col)
    field_map[friendly_name] = matches

with open(OUT / "07_resource_field_column_mapping.json", "w") as f:
    json.dump(field_map, f, indent=2)

# Create a concise block table that retains every matching source column.
block_table = valid_blocks[["_block_original", "_block_key"]].copy()

for friendly_name, columns in field_map.items():
    if not columns:
        block_table[friendly_name] = np.nan
    elif len(columns) == 1:
        block_table[friendly_name] = valid_blocks[columns[0]].values
    else:
        # Keep all subcolumns (often C, NC, PQ, Total) separately.
        for i, col in enumerate(columns, 1):
            block_table[f"{friendly_name}__source_col_{i}"] = (
                valid_blocks[col].values
            )

block_table.to_csv(
    OUT / "08_Alwar_blockwise_resource_assessment.csv",
    index=False
)

# -------------------------- 8. CHECK DUPLICATE SOURCE ROWS ------------

# Exact duplicate data rows in the Alwar GEC extract.
data_cols = [
    c for c in valid_blocks.columns
    if not c.startswith("_")
]

exact_dup_mask = valid_blocks.duplicated(
    subset=data_cols, keep=False
)
exact_duplicates = valid_blocks[exact_dup_mask].copy()

exact_duplicates.to_csv(
    OUT / "09_Alwar_exact_duplicate_assessment_rows.csv",
    index=False
)

# -------------------------- 9. BDR EVIDENCE QUALITY SUMMARY ------------

def safe_sum(df, col):
    if col not in df.columns:
        return None
    return pd.to_numeric(df[col], errors="coerce").sum(min_count=1)

bdr_summary = {
    "alwar_pdf_count_in_previous_scan": int(len(alwar_pdfs)),
    "alwar_pdf_processing_successes": int(
        (alwar_pdfs["processing_status"] == "SUCCESS").sum()
    ) if "processing_status" in alwar_pdfs.columns else None,
    "alwar_pdf_pages": safe_sum(alwar_pdfs, "page_count"),
    "alwar_pages_with_extracted_text": safe_sum(
        alwar_pdfs, "pages_with_extracted_text"
    ),
    "alwar_pages_using_ocr": safe_sum(alwar_pdfs, "pages_ocr_used"),
    "alwar_page_evidence_rows": int(len(page_evidence)),
    "alwar_candidate_parameter_lines": int(len(candidate_lines)),
    "alwar_gec_rows_extracted": int(len(alwar)),
    "alwar_nonblank_block_rows": int(len(valid_blocks)),
    "alwar_distinct_normalized_block_names": int(
        valid_blocks["_block_key"].nunique()
    ),
    "alwar_rows_in_duplicate_block_groups": int(len(duplicate_blocks)),
    "alwar_exact_duplicate_assessment_rows": int(len(exact_duplicates)),
    "hydraulic_parameters_verified_from_source_pages": False,
    "unique_georeferenced_boreholes_verified": False,
    "lithological_intervals_verified": False,
    "groundwater_heads_verified_for_calibration": False,
    "reason":
        "Automated PDF keyword screening and GEC spreadsheet extraction "
        "do not themselves verify unique boreholes, coordinates, lithological "
        "intervals, parameter units, or calibration-ready head observations."
}

with open(OUT / "10_Alwar_readiness_summary.json", "w") as f:
    json.dump(bdr_summary, f, indent=2, default=str)

# -------------------------- 10. DATA-READINESS TABLE ------------------

readiness_rows = [
    {
        "requirement": "Government resource assessment",
        "evidence_found": "GEC worksheet in State Report 2024-25",
        "status": "EXTRACTED — QA REQUIRED",
        "what_is_still_needed":
            "Verify column meanings, C/NC/PQ category splits, units, "
            "missing values, and block-level duplicates."
    },
    {
        "requirement": "Alwar BDR report coverage",
        "evidence_found": f"{len(alwar_pdfs)} PDFs in prior scan",
        "status": "SCREENED",
        "what_is_still_needed":
            "Review original pages and identify unique report records."
    },
    {
        "requirement": "Unique borehole inventory",
        "evidence_found": "Borehole-related keyword hits only",
        "status": "NOT YET VERIFIED",
        "what_is_still_needed":
            "Extract unique borehole IDs, coordinates, total depths, "
            "and source report/page; resolve cross-report duplicates."
    },
    {
        "requirement": "Lithological layer intervals",
        "evidence_found": "Lithology keyword evidence",
        "status": "NOT YET VERIFIED",
        "what_is_still_needed":
            "Extract top and bottom depths, lithology descriptions, "
            "and borehole coordinates for each interval."
    },
    {
        "requirement": "Hydraulic conductivity / transmissivity / storage",
        "evidence_found": "Candidate parameter lines from PDFs",
        "status": "NOT YET VERIFIED",
        "what_is_still_needed":
            "Verify numerical values, units, method, test location, "
            "aquifer interval and whether values are measured or inferred."
    },
    {
        "requirement": "Groundwater-head calibration observations",
        "evidence_found": "Separate cleaned monitoring dataset exists",
        "status": "AVAILABLE AS A SEPARATE SOURCE; MATCHING REQUIRED",
        "what_is_still_needed":
            "Match validated well IDs and coordinates, observation dates, "
            "water-level datum and measurement quality flags."
    },
    {
        "requirement": "Recharge and groundwater extraction stresses",
        "evidence_found": "GEC resource-assessment fields",
        "status": "EXTRACTED — INTERPRETATION REQUIRED",
        "what_is_still_needed":
            "Check reporting period, units, category-specific values, "
            "and whether values can be spatially assigned to the model."
    },
    {
        "requirement": "Boundary conditions and model domain",
        "evidence_found": "Not verified by this audit",
        "status": "NOT YET VERIFIED",
        "what_is_still_needed":
            "Map aquifer extent, rivers/canals, groundwater divides, "
            "geology and external boundary heads or fluxes."
    },
    {
        "requirement": "High-resolution grid justification",
        "evidence_found": "Not verified by this audit",
        "status": "NOT YET VERIFIED",
        "what_is_still_needed":
            "Use borehole spacing, aquifer heterogeneity, stress distribution "
            "and calibration density to choose cell size."
    }
]

readiness = pd.DataFrame(readiness_rows)
readiness.to_csv(
    OUT / "11_Alwar_verified_data_readiness_table.csv",
    index=False
)

# -------------------------- 11. PRINT RESULTS -------------------------

print("\n" + "=" * 75)
print("ALWAR DATA-READINESS AUDIT COMPLETED")
print("=" * 75)

for k, v in bdr_summary.items():
    print(f"{k}: {v}")

print("\nALWAR BLOCKS IN GEC ASSESSMENT")
print(
    valid_blocks[["_block_original"]]
    .drop_duplicates()
    .sort_values("_block_original")
    .to_string(index=False)
)

print("\nDUPLICATE BLOCK GROUPS — REVIEW, DO NOT AUTO-DELETE")
if duplicate_blocks.empty:
    print("No repeated normalized block names found.")
else:
    print(
        duplicate_blocks[["_block_original", "_block_key"]]
        .to_string(index=False)
    )

print("\nDATA-READINESS TABLE")
display(readiness)

print("\nBLOCK-WISE RESOURCE ASSESSMENT PREVIEW")
display(block_table.head(15))

print("\nOUTPUT FOLDER:")
print(OUT)

print("\nIMPORTANT:")
print(
    "This audit verifies the spreadsheet extract and screens PDF evidence. "
    "It does not claim that individual boreholes, lithological intervals, "
    "hydraulic parameters or calibration observations have been fully verified."
)

**[orig #15]** Step 2 — BDR borehole and lithology inventory  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 2: ALWAR BDR BOREHOLE & LITHOLOGY INVENTORY
# Google Colab | Source-preserving | Audit-oriented
#
# Reads:
#   1. Original Alwar BDR PDFs
#   2. Existing page-level BDR extracted-text CSV, if available
#
# Creates:
#   01_pdf_inventory.csv
#   02_page_text_inventory.csv
#   03_lithology_evidence.csv
#   04_borehole_and_location_evidence.csv
#   05_hydraulic_parameter_evidence.csv
#   06_manual_verification_register.csv
#   07_processing_summary.txt
#
# IMPORTANT:
# - Does not modify source PDFs or GEC workbook.
# - Keyword matches are evidence candidates, not verified boreholes.
# - Does not infer missing coordinates, layer boundaries or parameters.
# ================================================================

from pathlib import Path
import re
import sys
import subprocess
import pandas as pd

# ---------------------- USER PATHS -------------------------------

BDR_ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR"
)

EXISTING_EXTRACTION_DIR = Path(
    "/content/drive/MyDrive/Groundwater_BDR_Analysis_Output"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory"
)

# ---------------------- DRIVE MOUNT ------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab not detected; assuming Drive is already accessible.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------- DEPENDENCIES -----------------------------

try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pymupdf"]
    )
    import fitz

# ---------------------- DISCOVER ALWAR PDFS ----------------------

if not BDR_ROOT.exists():
    raise FileNotFoundError(
        f"BDR source folder not found: {BDR_ROOT}"
    )

pdf_files = [
    p for p in BDR_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".pdf"
    and "alwar" in str(p.parent).lower()
]

# Also include PDFs directly inside the root whose filenames
# explicitly identify Alwar.
pdf_files = sorted(set(
    pdf_files + [
        p for p in BDR_ROOT.glob("*.pdf")
        if "alwar" in p.name.lower()
    ]
))

if not pdf_files:
    raise FileNotFoundError(
        "No PDFs in the configured BDR folder were identified "
        "as Alwar reports. Check BDR_ROOT and folder names."
    )

# ---------------------- PDF INVENTORY ----------------------------

pdf_inventory = []
page_records = []
errors = []

for pdf_path in pdf_files:
    try:
        with fitz.open(pdf_path) as doc:
            page_count = len(doc)
            pages_with_text = 0
            pages_without_text = 0

            for page_number, page in enumerate(doc, start=1):
                text = page.get_text("text") or ""
                text = text.replace("\x00", " ")

                if text.strip():
                    pages_with_text += 1
                else:
                    pages_without_text += 1

                page_records.append({
                    "pdf_file": pdf_path.name,
                    "source_relative_path": str(
                        pdf_path.relative_to(BDR_ROOT)
                    ),
                    "source_page": page_number,
                    "extracted_text": text,
                    "extracted_character_count": len(text.strip()),
                    "text_extraction_status": (
                        "TEXT_EXTRACTED"
                        if text.strip()
                        else "NO_TEXT_LAYER"
                    )
                })

            pdf_inventory.append({
                "pdf_file": pdf_path.name,
                "source_relative_path": str(
                    pdf_path.relative_to(BDR_ROOT)
                ),
                "page_count": page_count,
                "pages_with_text": pages_with_text,
                "pages_without_text": pages_without_text,
                "pdf_processing_status": "PROCESSED"
            })

    except Exception as exc:
        errors.append({
            "source_file": str(pdf_path),
            "error": repr(exc)
        })

        pdf_inventory.append({
            "pdf_file": pdf_path.name,
            "source_relative_path": str(pdf_path),
            "page_count": None,
            "pages_with_text": None,
            "pages_without_text": None,
            "pdf_processing_status": "ERROR"
        })

pdf_inventory_df = pd.DataFrame(pdf_inventory)
page_df = pd.DataFrame(page_records)

# ---------------------- EVIDENCE PATTERNS ------------------------
# These patterns flag passages for review; they do not prove that
# a passage describes a unique borehole or a measured parameter.

lithology_pattern = re.compile(
    r"\blitholog(?:y|ical|ic)?\b|"
    r"\blitho[\s-]?log\b|"
    r"\bgeological\s+log\b|"
    r"\bbore[\s-]?log\b|"
    r"\bdrill(?:ing)?\s+log\b|"
    r"\bsoil\s+profile\b|"
    r"\bstrata\b|"
    r"\bclay\b|\bsilt\b|\bsand\b|"
    r"\bgravel\b|\bboulder(?:s)?\b|"
    r"\bweathered\s+rock\b|\bfractured\s+rock\b|"
    r"\bbasement\s+rock\b|\baquifer\b",
    re.IGNORECASE
)

borehole_pattern = re.compile(
    r"\bbore[\s-]?well\b|"
    r"\bbore[\s-]?hole\b|"
    r"\bdrill(?:ed|ing)?\s+well\b|"
    r"\bwell\s+number\b|\bwell\s+no\.?\b|"
    r"\btube[\s-]?well\b|"
    r"\bopen\s+well\b|\bdug\s+well\b|"
    r"\bexploratory\s+well\b|"
    r"\bexploratory\s+borehole\b",
    re.IGNORECASE
)

location_pattern = re.compile(
    r"\blatitude\b|\blongitude\b|\bcoordinates?\b|"
    r"\bGPS\b|\bUTM\b|\beasting\b|\bnorthing\b|"
    r"\bvillage\b|\bblock\b|\btehsil\b|\btehsil\b|"
    r"\bdistrict\b|\bsite\s+name\b|\blocation\b",
    re.IGNORECASE
)

hydraulic_pattern = re.compile(
    r"\bhydraulic\s+conductivity\b|"
    r"\bpermeability\b|"
    r"\btransmissivity\b|"
    r"\bstorativity\b|"
    r"\bstorage\s+coefficient\b|"
    r"\bspecific\s+yield\b|"
    r"\bspecific\s+capacity\b|"
    r"\bcoefficient\s+of\s+permeability\b|"
    r"\bpumping[\s-]?test\b|"
    r"\bslug\s+test\b|"
    r"\bK\s*=\s*[\d.]+",
    re.IGNORECASE
)

depth_pattern = re.compile(
    r"\bdepth\b|\bmbgl\b|\bm\s*bgl\b|"
    r"\bmet(?:re|er)s?\b|\bthickness\b|"
    r"\bfrom\b.{0,30}\bto\b",
    re.IGNORECASE
)

# ---------------------- EXTRACT CANDIDATE EVIDENCE ---------------

lithology_records = []
borehole_records = []
hydraulic_records = []
manual_records = []

for _, row in page_df.iterrows():
    text = str(row["extracted_text"])
    lines = text.splitlines()

    for line_number, line in enumerate(lines, start=1):
        line = line.strip()
        if not line:
            continue

        common = {
            "pdf_file": row["pdf_file"],
            "source_relative_path": row["source_relative_path"],
            "source_page": row["source_page"],
            "line_number": line_number,
            "evidence_text": line
        }

        if lithology_pattern.search(line):
            lithology_records.append({
                **common,
                "depth_term_detected": bool(
                    depth_pattern.search(line)
                ),
                "verification_status": "CANDIDATE_NOT_VERIFIED"
            })

        if borehole_pattern.search(line) or location_pattern.search(line):
            borehole_records.append({
                **common,
                "borehole_term_detected": bool(
                    borehole_pattern.search(line)
                ),
                "location_term_detected": bool(
                    location_pattern.search(line)
                ),
                "verification_status": "CANDIDATE_NOT_VERIFIED"
            })

        if hydraulic_pattern.search(line):
            hydraulic_records.append({
                **common,
                "parameter_term_detected": True,
                "verification_status": "VALUE_AND_UNIT_REQUIRE_REVIEW"
            })

# ---------------------- PAGE-LEVEL REVIEW REGISTER ---------------

for _, row in page_df.iterrows():
    text = str(row["extracted_text"])

    if not text.strip():
        reason = "NO_TEXT_LAYER; OCR OR IMAGE REVIEW MAY BE REQUIRED"
    elif (
        lithology_pattern.search(text)
        or borehole_pattern.search(text)
        or hydraulic_pattern.search(text)
    ):
        reason = "REVIEW_CANDIDATE_EVIDENCE_AND_SOURCE_PAGE"
    else:
        reason = "NO_TARGET_KEYWORD_DETECTED; NOT PROOF_OF_ABSENCE"

    manual_records.append({
        "pdf_file": row["pdf_file"],
        "source_relative_path": row["source_relative_path"],
        "source_page": row["source_page"],
        "review_reason": reason,
        "manual_review_status": "PENDING"
    })

# ---------------------- WRITE OUTPUTS ----------------------------

def save_csv(records, filename, columns=None):
    df = pd.DataFrame(records)
    if df.empty and columns:
        df = pd.DataFrame(columns=columns)
    df.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )
    return df

pdf_inventory_df.to_csv(
    OUTPUT_DIR / "01_pdf_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

page_df.to_csv(
    OUTPUT_DIR / "02_page_text_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

lithology_df = save_csv(
    lithology_records,
    "03_lithology_evidence.csv",
    [
        "pdf_file", "source_relative_path", "source_page",
        "line_number", "evidence_text", "depth_term_detected",
        "verification_status"
    ]
)

borehole_df = save_csv(
    borehole_records,
    "04_borehole_and_location_evidence.csv",
    [
        "pdf_file", "source_relative_path", "source_page",
        "line_number", "evidence_text", "borehole_term_detected",
        "location_term_detected", "verification_status"
    ]
)

hydraulic_df = save_csv(
    hydraulic_records,
    "05_hydraulic_parameter_evidence.csv",
    [
        "pdf_file", "source_relative_path", "source_page",
        "line_number", "evidence_text", "parameter_term_detected",
        "verification_status"
    ]
)

manual_df = save_csv(
    manual_records,
    "06_manual_verification_register.csv",
    [
        "pdf_file", "source_relative_path", "source_page",
        "review_reason", "manual_review_status"
    ]
)

pd.DataFrame(errors).to_csv(
    OUTPUT_DIR / "08_processing_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

# ---------------------- SUMMARY ----------------------------------

summary_lines = [
    "ALWAR BDR INVENTORY: STEP 2",
    "============================",
    f"Source folder: {BDR_ROOT}",
    f"Output folder: {OUTPUT_DIR}",
    f"PDF files discovered: {len(pdf_files)}",
    f"PDF files processed: {(pdf_inventory_df['pdf_processing_status'] == 'PROCESSED').sum()}",
    f"PDF processing errors: {len(errors)}",
    f"Pages with extracted text: {(page_df['text_extraction_status'] == 'TEXT_EXTRACTED').sum() if not page_df.empty else 0}",
    f"Pages without text layer: {(page_df['text_extraction_status'] == 'NO_TEXT_LAYER').sum() if not page_df.empty else 0}",
    f"Lithology evidence lines: {len(lithology_df)}",
    f"Borehole/location evidence lines: {len(borehole_df)}",
    f"Hydraulic parameter evidence lines: {len(hydraulic_df)}",
    "",
    "INTERPRETATION LIMITATIONS",
    "- Evidence lines are not counts of unique boreholes.",
    "- Keyword matches are not validated lithological intervals.",
    "- Hydraulic parameter values and units require source-page verification.",
    "- Pages without extracted text may contain scanned tables/figures.",
    "- Missing evidence does not establish that information is absent.",
    "- No coordinates, depths, parameters, or layer boundaries were invented.",
    "- Original source PDFs and GEC workbook were not modified."
]

summary_path = OUTPUT_DIR / "07_processing_summary.txt"
summary_path.write_text(
    "\n".join(summary_lines),
    encoding="utf-8"
)

print("\n".join(summary_lines))
print("\nCreated files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


**[orig #16]** Preview the inventory CSVs  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory"
)

files = [
    "03_lithology_evidence.csv",
    "04_borehole_and_location_evidence.csv",
    "05_hydraulic_parameter_evidence.csv",
]

for filename in files:
    path = folder / filename
    print("\n" + "=" * 90)
    print(filename)

    df = pd.read_csv(path, nrows=5, low_memory=False)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst five records:")
    with pd.option_context(
        "display.max_colwidth", 180,
        "display.max_columns", None,
        "display.width", 220
    ):
        display(df)


**[orig #17]** Step 3 — context-based extraction from BDR pages  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import re
import pandas as pd

# ============================================================
# STEP 3: CONTEXT-BASED ALWAR BDR EXTRACTION
# Original reports and GEC workbook remain unchanged.
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory"
)

INPUT = BASE / "02_page_text_inventory.csv"

OUT = BASE / "Step_3_Context_Extraction"
OUT.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise FileNotFoundError(f"Missing input file: {INPUT}")

pages = pd.read_csv(INPUT, low_memory=False)

required = {
    "pdf_file", "source_relative_path",
    "source_page", "extracted_text"
}
missing = required - set(pages.columns)

if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Match likely lithology, well, depth, and hydraulic terminology.
LITH = re.compile(
    r"litholog|litho[\s-]?log|strata|surface soil|"
    r"clay|kankar|sand|gravel|boulder|silt|"
    r"weathered rock|fractured rock|aquifer|"
    r"geological log|bore[\s-]?log",
    re.I
)

WELL = re.compile(
    r"bore[\s-]?hole|bore[\s-]?well|well\s*(?:no\.?|number)|"
    r"tube[\s-]?well|open well|dug well|village|"
    r"tehsil|latitude|longitude|coordinates|"
    r"total depth|depth of well",
    re.I
)

HYD = re.compile(
    r"hydraulic conductivity|permeability|transmissivity|"
    r"specific capacity|specific yield|storage coefficient|"
    r"storativity|pumping[\s-]?test|slug\s+test",
    re.I
)

DEPTH = re.compile(
    r"(?<![A-Za-z])\d+(?:\.\d+)?\s*"
    r"(?:m(?:et(?:er|re)s?)?|mbgl|m\s*bgl|metres?|meters?)\b",
    re.I
)

# Extract numeric tokens only as candidates. No assumption is made
# that every number is a depth or hydraulic parameter.
NUMBER = re.compile(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?")

context_rows = []
review_rows = []
depth_candidates = []
hydraulic_candidates = []

for _, row in pages.iterrows():
    raw = row["extracted_text"]
    text = "" if pd.isna(raw) else str(raw)
    lines = text.splitlines()

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        is_lith = bool(LITH.search(line))
        is_well = bool(WELL.search(line))
        is_hyd = bool(HYD.search(line))

        if not (is_lith or is_well or is_hyd):
            continue

        # Keep adjacent lines for context, without claiming that
        # they necessarily belong to the same table row.
        start = max(0, i - 2)
        end = min(len(lines), i + 3)
        context = " | ".join(
            x.strip() for x in lines[start:end] if x.strip()
        )

        common = {
            "pdf_file": row["pdf_file"],
            "source_relative_path": row["source_relative_path"],
            "source_page": row["source_page"],
            "matched_line_number": i + 1,
            "matched_line": line,
            "context_previous_current_next": context
        }

        categories = []
        if is_lith:
            categories.append("LITHOLOGY")
        if is_well:
            categories.append("BOREHOLE_OR_LOCATION")
        if is_hyd:
            categories.append("HYDRAULIC_PARAMETER")

        context_rows.append({
            **common,
            "evidence_categories": ";".join(categories),
            "explicit_depth_unit_detected": bool(DEPTH.search(line)),
            "verification_status": "UNVERIFIED_CANDIDATE"
        })

        review_reasons = []
        if is_lith:
            review_reasons.append("Verify lithology and interval boundaries")
        if is_well:
            review_reasons.append("Verify well identity and location")
        if is_hyd:
            review_reasons.append("Verify parameter value, unit and method")

        review_rows.append({
            **common,
            "review_reason": "; ".join(review_reasons),
            "review_status": "PENDING_MANUAL_SOURCE_CHECK"
        })

        # Numeric/depth candidates from lithology passages.
        if is_lith:
            depth_candidates.append({
                **common,
                "numeric_tokens": ";".join(NUMBER.findall(line)),
                "depth_unit_matches": ";".join(
                    m.group(0) for m in DEPTH.finditer(line)
                ),
                "top_depth_m": None,
                "bottom_depth_m": None,
                "interval_status": (
                    "EXPLICIT_DEPTH_TEXT_PRESENT_REQUIRES_VALIDATION"
                    if DEPTH.search(line)
                    else "NO_EXPLICIT_DEPTH_UNIT_IN_MATCHED_LINE"
                )
            })

        # Candidate hydraulic lines plus adjacent context.
        if is_hyd:
            hydraulic_candidates.append({
                **common,
                "numeric_tokens_in_context": ";".join(
                    NUMBER.findall(context)
                ),
                "unit_text_in_context": ";".join(
                    re.findall(
                        r"\b(?:m2?/day|m²/day|m/s|m/day|"
                        r"lps|l/s|m3/day|m³/day|percent|%)\b",
                        context,
                        flags=re.I
                    )
                ),
                "parameter_value": None,
                "parameter_unit": None,
                "validation_status": "VALUE_AND_UNIT_NOT_YET_CONFIRMED"
            })

# Save output tables.
def save(records, filename):
    df = pd.DataFrame(records)
    df.to_csv(OUT / filename, index=False, encoding="utf-8-sig")
    return df

context_df = save(
    context_rows, "01_context_evidence.csv"
)
depth_df = save(
    depth_candidates, "02_lithology_depth_candidates.csv"
)
hydraulic_df = save(
    hydraulic_candidates, "03_hydraulic_candidates.csv"
)
review_df = save(
    review_rows, "04_manual_verification_register.csv"
)

# Summarize counts; counts represent evidence lines, not unique wells.
summary = [
    "ALWAR BDR STEP 3 SUMMARY",
    "=========================",
    f"Pages in input inventory: {len(pages)}",
    f"Context evidence rows: {len(context_df)}",
    f"Lithology depth candidate rows: {len(depth_df)}",
    f"Hydraulic candidate rows: {len(hydraulic_df)}",
    f"Manual review rows: {len(review_df)}",
    "",
    "IMPORTANT",
    "- These are evidence lines, not counts of unique boreholes.",
    "- Adjacent text is provided as context, not guaranteed table structure.",
    "- Numeric tokens are not automatically assigned to depths or parameters.",
    "- Top and bottom interval depths remain blank until verified.",
    "- Hydraulic values and units remain unconfirmed until source checking.",
    "- Original PDFs and the GEC workbook were not modified."
]

(OUT / "05_summary.txt").write_text(
    "\n".join(summary), encoding="utf-8"
)

print("\n".join(summary))
print("\nOutput files:")
for p in sorted(OUT.iterdir()):
    print(p.name)

print("\nSample context evidence:")
display(context_df.head(10))


**[orig #18]** Show the extracted text for a given PDF page  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

source = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory/"
    "02_page_text_inventory.csv"
)

df = pd.read_csv(source, low_memory=False)

targets = [
    ("ALWAR BDR 3 2011 part 2.pdf", 2),
    ("ALWAR BDR 3 2011 part 2.pdf", 12),
]

for filename, page_number in targets:
    selected = df[
        df["pdf_file"].astype(str).eq(filename)
        & pd.to_numeric(
            df["source_page"], errors="coerce"
        ).eq(page_number)
    ]

    print("\n" + "=" * 90)
    print(f"FILE: {filename} | PDF PAGE: {page_number}")

    if selected.empty:
        print("Page not found in the inventory.")
        continue

    text = str(selected.iloc[0]["extracted_text"])

    for line_number, line in enumerate(text.splitlines(), start=1):
        print(f"{line_number:03d}: {line}")


## 2. OCR and visual review

**[orig #19]** Step 4 — Tesseract OCR of all Alwar BDR PDFs  
<sub>✅ ran OK</sub>

In [ ]:

# ============================================================
# STEP 4: OPEN-SOURCE OCR FOR ALL ALWAR BDR PDFs
# Engine: Tesseract OCR
# Runtime: Google Colab
#
# - OCRs every page in every discovered Alwar PDF
# - Preserves original PDFs and existing extracted text
# - Saves page-level OCR text and confidence summaries
# - Resumes from previously completed pages
# - Does not modify the GEC workbook
# ============================================================

from pathlib import Path
import subprocess
import sys
import json
import time
import re
import pandas as pd
import fitz
import pytesseract

from PIL import Image
from pytesseract import Output
from IPython.display import display

# -------------------- MOUNT GOOGLE DRIVE ----------------------

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Using existing filesystem.")

# -------------------- PATHS -----------------------------------

BDR_ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory/"
    "OpenSource_Tesseract_OCR"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OCR configuration
OCR_LANGUAGE = "eng"
RENDER_ZOOM = 2.5
TESSERACT_CONFIG = "--oem 3 --psm 6"

# -------------------- INSTALL TESSERACT -----------------------

if subprocess.run(
    ["bash", "-lc", "command -v tesseract"],
    capture_output=True,
    text=True
).returncode != 0:
    subprocess.run(
        ["bash", "-lc",
         "apt-get update -qq && apt-get install -y tesseract-ocr"],
        check=True
    )

print("Tesseract version:")
print(pytesseract.get_tesseract_version())

# -------------------- FIND ALWAR PDFS -------------------------

if not BDR_ROOT.exists():
    raise FileNotFoundError(
        f"BDR source directory does not exist: {BDR_ROOT}"
    )

pdf_files = sorted({
    p for p in BDR_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".pdf"
    and "alwar" in str(p.parent).lower()
} | {
    p for p in BDR_ROOT.glob("*.pdf")
    if "alwar" in p.name.lower()
})

if not pdf_files:
    raise FileNotFoundError(
        "No Alwar PDFs were found. Verify BDR_ROOT."
    )

print(f"\nPDF files discovered: {len(pdf_files)}")
for p in pdf_files:
    print(" -", p.relative_to(BDR_ROOT))

# -------------------- CHECKPOINT FILES ------------------------

def safe_name(path):
    return re.sub(
        r"[^A-Za-z0-9._-]+", "_", path.stem
    )

def jsonl_path(pdf_path):
    return OUTPUT_DIR / (
        safe_name(pdf_path) + "_ocr_pages.jsonl"
    )

# Each completed page is stored as one JSON line.
# This allows the run to resume after interruption.

# -------------------- OCR EACH PAGE ---------------------------

run_summary = []

for pdf_index, pdf_path in enumerate(pdf_files, start=1):
    relative_path = str(pdf_path.relative_to(BDR_ROOT))
    checkpoint = jsonl_path(pdf_path)

    completed_pages = set()

    if checkpoint.exists():
        with open(checkpoint, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    completed_pages.add(int(record["source_page"]))
                except Exception:
                    continue

    print(
        f"\n[{pdf_index}/{len(pdf_files)}] {pdf_path.name}"
    )

    errors = []
    start_time = time.time()

    try:
        doc = fitz.open(pdf_path)
    except Exception as exc:
        run_summary.append({
            "pdf_file": pdf_path.name,
            "source_relative_path": relative_path,
            "status": "PDF_OPEN_FAILED",
            "total_pages": None,
            "pages_previously_completed": 0,
            "pages_ocr_completed_this_run": 0,
            "errors": str(exc)
        })
        continue

    total_pages = len(doc)
    completed_this_run = 0

    try:
        with open(
            checkpoint, "a", encoding="utf-8"
        ) as out_file:

            for page_index in range(total_pages):
                page_number = page_index + 1

                if page_number in completed_pages:
                    continue

                try:
                    page = doc[page_index]

                    # Render original page without changing the PDF.
                    pix = page.get_pixmap(
                        matrix=fitz.Matrix(
                            RENDER_ZOOM, RENDER_ZOOM
                        ),
                        alpha=False
                    )

                    image = Image.frombytes(
                        "RGB",
                        [pix.width, pix.height],
                        pix.samples
                    )

                    # OCR text.
                    ocr_text = pytesseract.image_to_string(
                        image,
                        lang=OCR_LANGUAGE,
                        config=TESSERACT_CONFIG
                    )

                    # Word-level confidence information.
                    # Tesseract may return -1 for non-word regions.
                    data = pytesseract.image_to_data(
                        image,
                        lang=OCR_LANGUAGE,
                        config=TESSERACT_CONFIG,
                        output_type=Output.DATAFRAME
                    )

                    valid_conf = pd.to_numeric(
                        data["conf"], errors="coerce"
                    )

                    valid_conf = valid_conf[
                        (valid_conf >= 0)
                        & data["text"].fillna("").astype(str).str.strip().ne("")
                    ]

                    mean_confidence = (
                        float(valid_conf.mean())
                        if len(valid_conf) else None
                    )

                    record = {
                        "pdf_file": pdf_path.name,
                        "source_relative_path": relative_path,
                        "source_page": page_number,
                        "ocr_engine": "Tesseract",
                        "ocr_language": OCR_LANGUAGE,
                        "ocr_config": TESSERACT_CONFIG,
                        "render_zoom": RENDER_ZOOM,
                        "image_width_px": pix.width,
                        "image_height_px": pix.height,
                        "ocr_text": ocr_text,
                        "ocr_character_count": len(ocr_text.strip()),
                        "ocr_word_count": int(len(valid_conf)),
                        "mean_word_confidence": mean_confidence,
                        "ocr_status": (
                            "TEXT_RETURNED"
                            if ocr_text.strip()
                            else "NO_TEXT_RETURNED"
                        )
                    }

                    out_file.write(
                        json.dumps(
                            record, ensure_ascii=False
                        ) + "\n"
                    )
                    out_file.flush()

                    completed_pages.add(page_number)
                    completed_this_run += 1

                    if (
                        page_number % 25 == 0
                        or page_number == total_pages
                    ):
                        print(
                            f"  Page {page_number}/{total_pages} | "
                            f"completed this run: {completed_this_run}"
                        )

                except Exception as exc:
                    errors.append({
                        "source_page": page_number,
                        "error": repr(exc)
                    })
                    print(
                        f"  ERROR on page {page_number}: {exc}"
                    )

    finally:
        doc.close()

    # Save errors for this PDF separately.
    error_path = OUTPUT_DIR / (
        safe_name(pdf_path) + "_ocr_errors.json"
    )
    with open(error_path, "w", encoding="utf-8") as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)

    run_summary.append({
        "pdf_file": pdf_path.name,
        "source_relative_path": relative_path,
        "status": (
            "COMPLETED"
            if len(completed_pages) == total_pages and not errors
            else "PARTIAL_OR_ERRORS"
        ),
        "total_pages": total_pages,
        "pages_completed_total": len(completed_pages),
        "pages_ocr_completed_this_run": completed_this_run,
        "pages_with_errors": len(errors),
        "elapsed_seconds_this_run": round(
            time.time() - start_time, 1
        ),
        "checkpoint_file": checkpoint.name
    })

    pd.DataFrame(run_summary).to_csv(
        OUTPUT_DIR / "00_processing_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

# -------------------- CONSOLIDATE PAGE OUTPUTS ----------------

all_records = []

for pdf_path in pdf_files:
    checkpoint = jsonl_path(pdf_path)

    if not checkpoint.exists():
        continue

    with open(checkpoint, "r", encoding="utf-8") as f:
        for line in f:
            try:
                all_records.append(json.loads(line))
            except Exception:
                pass

ocr_df = pd.DataFrame(all_records)

if not ocr_df.empty:
    ocr_df = ocr_df.sort_values(
        ["source_relative_path", "source_page"]
    )

ocr_df.to_csv(
    OUTPUT_DIR / "01_all_pages_ocr_text.csv",
    index=False,
    encoding="utf-8-sig"
)

# Create a plain text export for easier inspection.
text_path = OUTPUT_DIR / "02_all_pages_ocr_text.txt"

with open(text_path, "w", encoding="utf-8") as f:
    for _, row in ocr_df.iterrows():
        f.write("\n" + "=" * 90 + "\n")
        f.write(
            f"FILE: {row['pdf_file']} | "
            f"PDF PAGE: {row['source_page']}\n"
        )
        f.write("=" * 90 + "\n")
        f.write(str(row["ocr_text"]))
        f.write("\n")

# Low-confidence pages are review candidates, not automatic failures.
if not ocr_df.empty:
    low_conf = ocr_df[
        ocr_df["mean_word_confidence"].isna()
        | (ocr_df["mean_word_confidence"] < 70)
    ].copy()
else:
    low_conf = pd.DataFrame()

low_conf.to_csv(
    OUTPUT_DIR / "03_low_confidence_pages.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== FINAL OCR SUMMARY ==========")
print(f"PDFs discovered: {len(pdf_files)}")
print(f"Page records saved: {len(ocr_df)}")
print(f"Low-confidence review candidates: {len(low_conf)}")
print(f"Output folder: {OUTPUT_DIR}")

if not ocr_df.empty:
    print("\nOCR records by PDF:")
    display(
        ocr_df.groupby("pdf_file").agg(
            pages=("source_page", "nunique"),
            mean_confidence=("mean_word_confidence", "mean"),
            empty_pages=(
                "ocr_status",
                lambda s: int((s == "NO_TEXT_RETURNED").sum())
            )
        ).reset_index()
    )

print("\nOriginal PDFs and GEC workbook were not modified.")


**[orig #20]** Step 4 — compare OCR vs text layer; quality control  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 4 — ALWAR BDR OCR COMPARISON & QUALITY CONTROL
# Complete Google Colab script
#
# Compares:
#   1. Tesseract OCR output
#   2. Existing PDF text extraction
#
# OUTPUT:
#   01_page_by_page_comparison.csv
#   02_prioritized_review_pages.csv
#   03_numeric_difference_candidates.csv
#   04_hydrogeology_page_candidates.csv
#   05_manual_verification_register.csv
#   06_comparison_summary.txt
#
# Original source files are read-only.
# No unverified values are accepted as confirmed data.
# ================================================================

from pathlib import Path
import re
import difflib
import numpy as np
import pandas as pd
import importlib.util

# ================================================================
# 1. GOOGLE DRIVE
# ================================================================

if importlib.util.find_spec("google.colab") is not None:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

DRIVE = Path("/content/drive/MyDrive")

OCR_DIR = (
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "OpenSource_Tesseract_OCR"
)

OCR_FILE = OCR_DIR / "01_all_pages_ocr_text.csv"

SOURCE_CANDIDATES = [
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "02_page_text_inventory.csv",

    DRIVE / "Groundwater_BDR_Analysis_Output"
    / "02_page_level_extracted_text.csv",
]

OUT_DIR = (
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "Comparison_QC"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not OCR_FILE.exists():
    raise FileNotFoundError(
        f"OCR file not found:\n{OCR_FILE}\n"
        "Confirm that the previous Tesseract OCR run completed."
    )

# ================================================================
# 2. HELPER FUNCTIONS
# ================================================================

def norm_col(value):
    """Normalize a column name for matching."""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def find_column(df, aliases, required=True):
    """
    Find a column using known aliases.
    Exact normalized matches are preferred over partial matches.
    """
    normalized = {norm_col(c): c for c in df.columns}

    for alias in aliases:
        key = norm_col(alias)
        if key in normalized:
            return normalized[key]

    # Conservative partial matching
    for alias in aliases:
        key = norm_col(alias)
        if len(key) < 5:
            continue

        matches = [
            c for c in df.columns
            if key in norm_col(c)
        ]

        if len(matches) == 1:
            return matches[0]

    if required:
        raise ValueError(
            f"Could not identify a required column.\n"
            f"Accepted aliases: {aliases}\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


def normalize_pdf_name(value):
    """Match using the PDF basename, not the folder path."""
    if pd.isna(value):
        return ""

    value = str(value).replace("\\", "/").split("/")[-1]
    value = re.sub(r"\s+", " ", value.strip())

    return value.casefold()


def page_to_int(value):
    """Convert a page field to an integer page number."""
    if pd.isna(value):
        return np.nan

    # Handle numeric values and text such as 'Page 12'
    try:
        numeric = float(str(value).strip())
        if numeric.is_integer():
            return int(numeric)
    except (ValueError, TypeError):
        pass

    match = re.search(r"\d+", str(value))
    return int(match.group()) if match else np.nan


def normalize_text(value):
    """Normalize whitespace and case without changing source files."""
    if pd.isna(value):
        return ""

    text = str(value).casefold()
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def numeric_tokens(value):
    """
    Extract number-like strings in their original sequence.
    These are review indicators, not confirmed measurements.
    """
    if pd.isna(value):
        return []

    text = str(value).replace(",", "")

    return re.findall(
        r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?",
        text
    )


def hydrogeology_flags(text):
    """Detect pages that may contain relevant hydrogeological evidence."""
    t = normalize_text(text)

    terms = {
        "lithology": [
            "lithology", "lithological", "formation",
            "clay", "kankar", "quartzite", "sandstone",
            "gravel", "boulder", "weathered", "fractured"
        ],
        "borehole_location": [
            "bore hole", "borehole", "village", "tehsil",
            "district", "latitude", "longitude",
            "panchayat samiti", "location"
        ],
        "depth_interval": [
            "depth", "thickness", "from", "to",
            "mbgl", "m bgl", "meter", "metre"
        ],
        "hydraulic_test": [
            "pumping test", "drawdown", "recuperation",
            "transmissivity", "permeability", "hydraulic",
            "specific capacity", "discharge", "yield",
            "storativity", "conductivity"
        ],
        "well_construction": [
            "pipe assembly", "slotted pipe", "plain pipe",
            "casing", "screen", "gravel pack"
        ],
    }

    return {
        key: any(term in t for term in values)
        for key, values in terms.items()
    }


def make_similarity(a, b):
    """Return a character-sequence similarity ratio."""
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return np.nan

    return difflib.SequenceMatcher(
        None, a, b, autojunk=False
    ).ratio()


# ================================================================
# 3. LOAD TESSERACT OCR
# ================================================================

ocr = pd.read_csv(OCR_FILE, low_memory=False)

print("=" * 70)
print("TESSERACT OCR INPUT")
print("=" * 70)
print("File:", OCR_FILE)
print("Rows:", len(ocr))
print("Columns:", list(ocr.columns))

# Actual names in your OCR file include:
# pdf_file, source_page, ocr_text, mean_word_confidence

ocr_file_col = find_column(
    ocr,
    [
        "pdf_file", "filename", "file_name",
        "source_pdf", "pdf_path", "document_name"
    ]
)

ocr_page_col = find_column(
    ocr,
    [
        "page_number", "page_num", "page",
        "page_no", "source_page"
    ]
)

ocr_text_col = find_column(
    ocr,
    [
        "ocr_text", "tesseract_text", "extracted_text",
        "page_text", "full_text", "text"
    ]
)

ocr_conf_col = find_column(
    ocr,
    [
        "mean_confidence", "mean_word_confidence",
        "ocr_confidence", "confidence",
        "average_confidence"
    ],
    required=False
)

ocr_std = pd.DataFrame({
    "pdf_file_ocr": ocr[ocr_file_col].astype(str),
    "pdf_key": ocr[ocr_file_col].map(normalize_pdf_name),
    "page_number": ocr[ocr_page_col].map(page_to_int),
    "ocr_text": ocr[ocr_text_col].fillna("").astype(str),
})

if ocr_conf_col:
    ocr_std["ocr_confidence"] = pd.to_numeric(
        ocr[ocr_conf_col], errors="coerce"
    )
else:
    ocr_std["ocr_confidence"] = np.nan

ocr_std = ocr_std.dropna(
    subset=["page_number"]
).copy()

ocr_std["page_number"] = ocr_std["page_number"].astype(int)

# Record duplicate PDF/page keys for quality control.
ocr_duplicate_count = int(
    ocr_std.duplicated(
        subset=["pdf_key", "page_number"], keep=False
    ).sum()
)

# If duplicate rows exist, combine their text rather than discarding it.
ocr_std = (
    ocr_std.groupby(
        ["pdf_key", "page_number"],
        as_index=False,
        dropna=False
    )
    .agg({
        "pdf_file_ocr": "first",
        "ocr_text": lambda s: "\n".join(
            value for value in s.astype(str) if value.strip()
        ),
        "ocr_confidence": "mean",
    })
)

print("\nOCR page records after key cleanup:", len(ocr_std))
print("PDFs:", ocr_std["pdf_key"].nunique())
print("Duplicate OCR rows before grouping:", ocr_duplicate_count)
print("Confidence field:", ocr_conf_col)

# ================================================================
# 4. FIND AND LOAD EXISTING PDF TEXT EXTRACTION
# ================================================================

source_std = None
source_path_used = None
source_report = []
source_best_overlap = -1

ocr_keys = set(
    zip(ocr_std["pdf_key"], ocr_std["page_number"])
)

for source_path in SOURCE_CANDIDATES:

    if not source_path.exists():
        source_report.append(
            f"NOT FOUND: {source_path}"
        )
        continue

    try:
        df = pd.read_csv(source_path, low_memory=False)

        source_report.append(
            f"FOUND: {source_path}; "
            f"rows={len(df)}; columns={list(df.columns)}"
        )

        file_col = find_column(
            df,
            [
                "pdf_file", "filename", "file_name",
                "source_pdf", "pdf_path", "document_name"
            ],
            required=False
        )

        page_col = find_column(
            df,
            [
                "page_number", "page_num", "page",
                "page_no", "source_page"
            ],
            required=False
        )

        text_col = find_column(
            df,
            [
                "page_text", "extracted_text", "full_text",
                "text", "page_content", "extracted_page_text"
            ],
            required=False
        )

        if file_col is None or page_col is None or text_col is None:
            source_report.append(
                "SKIPPED: could not identify all required "
                "PDF filename, page number and text columns."
            )
            continue

        candidate = pd.DataFrame({
            "pdf_file_source": df[file_col].astype(str),
            "pdf_key": df[file_col].map(normalize_pdf_name),
            "page_number": df[page_col].map(page_to_int),
            "source_text": df[text_col].fillna("").astype(str),
        })

        candidate = candidate.dropna(
            subset=["page_number"]
        ).copy()

        candidate["page_number"] = (
            candidate["page_number"].astype(int)
        )

        candidate = (
            candidate.groupby(
                ["pdf_key", "page_number"],
                as_index=False,
                dropna=False
            )
            .agg({
                "pdf_file_source": "first",
                "source_text": lambda s: "\n".join(
                    value for value in s.astype(str)
                    if value.strip()
                ),
            })
        )

        candidate_keys = set(
            zip(candidate["pdf_key"], candidate["page_number"])
        )

        overlap = len(ocr_keys & candidate_keys)

        print(
            f"\nCandidate source: {source_path}\n"
            f"  Usable PDF/page records: {len(candidate)}\n"
            f"  Matching OCR page keys: {overlap}"
        )

        if overlap > source_best_overlap:
            source_std = candidate
            source_path_used = source_path
            source_best_overlap = overlap

    except Exception as exc:
        source_report.append(
            f"ERROR: {source_path}: {exc}"
        )

if source_std is None:
    raise RuntimeError(
        "No usable existing text extraction was found.\n"
        "Inspect the source CSV columns below and update the "
        "column aliases in Section 4.\n\n"
        + "\n".join(source_report)
    )

print("\nSelected source:", source_path_used)
print("Source page records:", len(source_std))
print("Matching PDF/page keys:", source_best_overlap)

# ================================================================
# 5. JOIN OCR AND EXISTING EXTRACTION
# ================================================================

comparison = ocr_std.merge(
    source_std,
    on=["pdf_key", "page_number"],
    how="outer",
    indicator=True
)

comparison["pdf_file"] = (
    comparison["pdf_file_ocr"]
    .fillna(comparison["pdf_file_source"])
)

comparison["ocr_text"] = comparison["ocr_text"].fillna("")
comparison["source_text"] = comparison["source_text"].fillna("")

comparison["ocr_text_length"] = comparison["ocr_text"].str.len()
comparison["source_text_length"] = comparison["source_text"].str.len()

comparison["text_similarity_ratio"] = comparison.apply(
    lambda r: make_similarity(
        r["ocr_text"], r["source_text"]
    ),
    axis=1
)

comparison["text_length_ratio"] = comparison.apply(
    lambda r: (
        min(r["ocr_text_length"], r["source_text_length"])
        / max(r["ocr_text_length"], r["source_text_length"])
        if max(r["ocr_text_length"], r["source_text_length"]) > 0
        else np.nan
    ),
    axis=1
)

comparison["ocr_numeric_tokens"] = comparison["ocr_text"].map(
    numeric_tokens
)

comparison["source_numeric_tokens"] = comparison["source_text"].map(
    numeric_tokens
)

comparison["ocr_numeric_token_count"] = (
    comparison["ocr_numeric_tokens"].map(len)
)

comparison["source_numeric_token_count"] = (
    comparison["source_numeric_tokens"].map(len)
)

comparison["numeric_token_sequence_differs"] = comparison.apply(
    lambda r: (
        r["ocr_numeric_tokens"] != r["source_numeric_tokens"]
        if r["_merge"] == "both"
        else True
    ),
    axis=1
)

comparison["source_match_status"] = comparison["_merge"].map({
    "both": "Matched",
    "left_only": "OCR page only — source page not matched",
    "right_only": "Source page only — OCR page not matched",
})

# ================================================================
# 6. FLAG HYDROGEOLOGY-RELEVANT PAGES
# ================================================================

flags = comparison.apply(
    lambda r: hydrogeology_flags(
        str(r["ocr_text"]) + "\n" + str(r["source_text"])
    ),
    axis=1
)

for key in [
    "lithology",
    "borehole_location",
    "depth_interval",
    "hydraulic_test",
    "well_construction",
]:
    comparison[f"contains_{key}_terms"] = flags.map(
        lambda d: d[key]
    )

flag_columns = [
    "contains_lithology_terms",
    "contains_borehole_location_terms",
    "contains_depth_interval_terms",
    "contains_hydraulic_test_terms",
    "contains_well_construction_terms",
]

comparison["hydrogeology_relevance"] = (
    comparison[flag_columns].fillna(False).astype(int).sum(axis=1)
)

# ================================================================
# 7. REVIEW SCORING
# ================================================================

def score_page(row):
    score = 0
    reasons = []

    if row["source_match_status"] != "Matched":
        score += 5
        reasons.append("OCR/source page not matched")

    confidence = row["ocr_confidence"]

    if pd.notna(confidence):
        if confidence < 50:
            score += 5
            reasons.append("OCR confidence below 50")
        elif confidence < 70:
            score += 3
            reasons.append("OCR confidence below 70")
        elif confidence < 80:
            score += 1
            reasons.append("OCR confidence below 80")

    similarity = row["text_similarity_ratio"]

    if pd.notna(similarity):
        if similarity < 0.30:
            score += 5
            reasons.append("Very low text similarity")
        elif similarity < 0.55:
            score += 3
            reasons.append("Low text similarity")
        elif similarity < 0.75:
            score += 1
            reasons.append("Moderate text similarity")

    if bool(row["numeric_token_sequence_differs"]):
        score += 3
        reasons.append("Numeric token sequence differs")

    if (
        row["ocr_text_length"] == 0
        or row["source_text_length"] == 0
    ):
        score += 3
        reasons.append("One extraction is empty")

    relevance = row["hydrogeology_relevance"]

    if relevance >= 3:
        score += 3
        reasons.append("High hydrogeology keyword relevance")
    elif relevance >= 1:
        score += 1
        reasons.append("Hydrogeology terms present")

    return pd.Series({
        "review_score": score,
        "review_reasons": (
            "; ".join(reasons)
            if reasons
            else "No automatic priority trigger"
        ),
    })


scoring = comparison.apply(score_page, axis=1)
comparison = pd.concat([comparison, scoring], axis=1)

def priority_label(score):
    if score >= 10:
        return "Critical"
    elif score >= 6:
        return "High"
    elif score >= 3:
        return "Moderate"
    return "Low"

comparison["review_priority"] = (
    comparison["review_score"].map(priority_label)
)

# ================================================================
# 8. PREPARE COMPARISON OUTPUT
# ================================================================

output_columns = [
    "pdf_file",
    "page_number",
    "source_match_status",
    "ocr_confidence",
    "text_similarity_ratio",
    "text_length_ratio",
    "ocr_text_length",
    "source_text_length",
    "ocr_numeric_token_count",
    "source_numeric_token_count",
    "numeric_token_sequence_differs",
    "contains_lithology_terms",
    "contains_borehole_location_terms",
    "contains_depth_interval_terms",
    "contains_hydraulic_test_terms",
    "contains_well_construction_terms",
    "hydrogeology_relevance",
    "review_score",
    "review_priority",
    "review_reasons",
    "ocr_text",
    "source_text",
]

comparison_out = comparison[output_columns].copy()

comparison_out = comparison_out.sort_values(
    [
        "review_score",
        "hydrogeology_relevance",
        "pdf_file",
        "page_number",
    ],
    ascending=[False, False, True, True],
    na_position="last",
)

comparison_file = (
    OUT_DIR / "01_page_by_page_comparison.csv"
)

comparison_out.to_csv(
    comparison_file,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 9. PRIORITIZED REVIEW PAGES
# ================================================================

priority = comparison_out[
    comparison_out["review_priority"].isin(["Critical", "High"])
    | (comparison_out["hydrogeology_relevance"] >= 2)
].copy()

priority_file = OUT_DIR / "02_prioritized_review_pages.csv"

priority.to_csv(
    priority_file,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 10. NUMERIC DIFFERENCE CANDIDATES
# ================================================================

numeric_diff = comparison_out[
    comparison_out["numeric_token_sequence_differs"] == True
].copy()

numeric_file = (
    OUT_DIR / "03_numeric_difference_candidates.csv"
)

numeric_diff.to_csv(
    numeric_file,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 11. HYDROGEOLOGY PAGE CANDIDATES
# ================================================================

hydro = comparison_out[
    comparison_out["hydrogeology_relevance"] >= 1
].copy()

hydro_file = (
    OUT_DIR / "04_hydrogeology_page_candidates.csv"
)

hydro.to_csv(
    hydro_file,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 12. MANUAL VERIFICATION REGISTER
# ================================================================

manual_columns = [
    "pdf_file",
    "page_number",
    "review_priority",
    "review_score",
    "review_reasons",
    "ocr_confidence",
    "text_similarity_ratio",
    "numeric_token_sequence_differs",
    "contains_lithology_terms",
    "contains_borehole_location_terms",
    "contains_depth_interval_terms",
    "contains_hydraulic_test_terms",
    "contains_well_construction_terms",
]

manual = priority[manual_columns].copy()

manual["verified_by_human"] = ""
manual["borehole_id_verified"] = ""
manual["village_location_verified"] = ""
manual["lithology_description_verified"] = ""
manual["depth_from_m_verified"] = ""
manual["depth_to_m_verified"] = ""
manual["thickness_m_verified"] = ""
manual["hydraulic_parameter_verified"] = ""
manual["source_image_checked"] = ""
manual["reviewer_notes"] = ""
manual["verification_status"] = "Pending manual verification"

manual_file = (
    OUT_DIR / "05_manual_verification_register.csv"
)

manual.to_csv(
    manual_file,
    index=False,
    encoding="utf-8-sig"
)

# ================================================================
# 13. SUMMARY REPORT
# ================================================================

n_ocr = len(ocr_std)
n_source = len(source_std)

n_matched = int((comparison["_merge"] == "both").sum())
n_ocr_only = int((comparison["_merge"] == "left_only").sum())
n_source_only = int((comparison["_merge"] == "right_only").sum())

n_critical = int(
    (comparison_out["review_priority"] == "Critical").sum()
)

n_high = int(
    (comparison_out["review_priority"] == "High").sum()
)

summary_lines = [
    "ALWAR BDR OCR COMPARISON AND QUALITY CONTROL",
    "=" * 60,
    f"OCR input: {OCR_FILE}",
    f"Existing text source: {source_path_used}",
    f"Output folder: {OUT_DIR}",
    "",
    f"OCR PDF/page records: {n_ocr}",
    f"Existing extraction PDF/page records: {n_source}",
    f"Matched PDF/page keys: {n_matched}",
    f"OCR-only page keys: {n_ocr_only}",
    f"Source-only page keys: {n_source_only}",
    f"Critical review pages: {n_critical}",
    f"High review pages: {n_high}",
    f"Prioritized review register rows: {len(priority)}",
    f"Numeric difference candidates: {len(numeric_diff)}",
    f"Hydrogeology keyword candidate pages: {len(hydro)}",
    f"Manual verification register rows: {len(manual)}",
    "",
    "INTERPRETATION:",
    "- Similarity is a screening measure, not a correctness score.",
    "- Numeric differences may arise from OCR errors or layout differences.",
    "- Keyword hits do not verify the presence or validity of parameters.",
    "- Verify table rows, columns, units and borehole associations",
    "  against original PDF page images.",
    "- Do not use unverified candidates directly as model inputs.",
    "- Original PDFs, source extraction, OCR input and GEC workbook",
    "  were not modified by this script.",
    "",
    "SOURCE FILE DISCOVERY:",
    *source_report,
]

summary_file = OUT_DIR / "06_comparison_summary.txt"

summary_file.write_text(
    "\n".join(summary_lines),
    encoding="utf-8"
)

# ================================================================
# 14. DISPLAY RESULTS
# ================================================================

print("\n" + "=" * 70)
print("STEP 4 — COMPARISON COMPLETE")
print("=" * 70)

print("Selected source:", source_path_used)
print("Matched PDF/page keys:", n_matched)
print("OCR-only page keys:", n_ocr_only)
print("Source-only page keys:", n_source_only)
print("Critical review pages:", n_critical)
print("High review pages:", n_high)
print("Numeric difference candidates:", len(numeric_diff))
print("Hydrogeology candidate pages:", len(hydro))
print("Manual verification rows:", len(manual))

print("\nOutput folder:")
print(OUT_DIR)

print("\nFiles created:")
for path in [
    comparison_file,
    priority_file,
    numeric_file,
    hydro_file,
    manual_file,
    summary_file,
]:
    print(" -", path.name)

print("\nTop 20 pages for review:")

try:
    from IPython.display import display

    display(
        comparison_out[
            [
                "pdf_file",
                "page_number",
                "review_priority",
                "review_score",
                "ocr_confidence",
                "text_similarity_ratio",
                "numeric_token_sequence_differs",
                "hydrogeology_relevance",
                "review_reasons",
            ]
        ].head(20)
    )
except ImportError:
    print(
        comparison_out[
            [
                "pdf_file",
                "page_number",
                "review_priority",
                "review_score",
                "ocr_confidence",
                "text_similarity_ratio",
                "review_reasons",
            ]
        ].head(20).to_string(index=False)
    )

print("\nNEXT ACTION:")
print(
    "Review 02_prioritized_review_pages.csv. Compare the highest-priority "
    "pages with their original PDF page images before accepting any "
    "borehole IDs, lithology intervals or hydraulic parameters."
)


**[orig #21]** Step 5 — visual review packet (page images + contact sheets)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# STEP 5 — VISUAL REVIEW PACKET FOR ALWAR BDR
# Creates original-page images and contact sheets for manual review.
# Does not modify PDFs or any existing extracted data.
# ================================================================

from pathlib import Path
import re
import math
import pandas as pd
import fitz
from PIL import Image, ImageDraw, ImageFont

# -------------------- Paths --------------------
DRIVE = Path("/content/drive/MyDrive")

PDF_ROOT = DRIVE / "gw_framework" / "data" / "BDR" / "Alwar"

COMPARISON_FILE = (
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "Comparison_QC"
    / "01_page_by_page_comparison.csv"
)

OUT_DIR = (
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "Visual_Review_Packet"
)

PAGE_DIR = OUT_DIR / "Original_Page_Images"
SHEET_DIR = OUT_DIR / "Contact_Sheets"

PAGE_DIR.mkdir(parents=True, exist_ok=True)
SHEET_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- Settings --------------------
MAX_PAGES = 30          # Number of pages to render in this batch
RENDER_ZOOM = 1.8       # Higher values produce larger images
THUMBNAIL_WIDTH = 650
SHEET_COLUMNS = 2
SHEET_ROWS = 2          # Four pages per contact sheet

# -------------------- Check inputs --------------------
if not COMPARISON_FILE.exists():
    raise FileNotFoundError(
        f"Comparison CSV not found: {COMPARISON_FILE}"
    )

if not PDF_ROOT.exists():
    raise FileNotFoundError(
        f"Alwar PDF folder not found: {PDF_ROOT}"
    )

df = pd.read_csv(COMPARISON_FILE, low_memory=False)

required = [
    "pdf_file", "page_number", "review_score",
    "review_priority", "hydrogeology_relevance"
]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns in comparison CSV: {missing}")

df["page_number"] = pd.to_numeric(
    df["page_number"], errors="coerce"
)
df["review_score"] = pd.to_numeric(
    df["review_score"], errors="coerce"
)
df["hydrogeology_relevance"] = pd.to_numeric(
    df["hydrogeology_relevance"], errors="coerce"
)

df = df.dropna(
    subset=["pdf_file", "page_number"]
).copy()

df["page_number"] = df["page_number"].astype(int)

# -------------------- Locate original PDFs --------------------
pdf_lookup = {}

for path in PDF_ROOT.rglob("*.pdf"):
    pdf_lookup[path.name.casefold()] = path

print("PDFs found:", len(pdf_lookup))
print("PDF folder:", PDF_ROOT)

# -------------------- Prioritize review pages --------------------
# Prioritize relevance and score, but avoid rendering duplicate keys.
# Numeric differences alone are not treated as proof of OCR error.
df = df.sort_values(
    ["review_score", "hydrogeology_relevance"],
    ascending=[False, False],
    na_position="last"
)

selected = df.drop_duplicates(
    subset=["pdf_file", "page_number"]
).head(MAX_PAGES).copy()

if selected.empty:
    raise RuntimeError("No review pages were selected.")

# -------------------- Render original PDF pages --------------------
manifest = []
rendered_images = []

for idx, row in enumerate(selected.itertuples(index=False), start=1):
    pdf_name = str(row.pdf_file)
    page_num = int(row.page_number)
    pdf_path = pdf_lookup.get(pdf_name.casefold())

    if pdf_path is None:
        manifest.append({
            "selection_order": idx,
            "pdf_file": pdf_name,
            "page_number": page_num,
            "image_path": "",
            "status": "PDF_NOT_FOUND",
            "review_score": getattr(row, "review_score", None),
            "hydrogeology_relevance": getattr(
                row, "hydrogeology_relevance", None
            ),
        })
        print("PDF not found:", pdf_name)
        continue

    try:
        with fitz.open(pdf_path) as doc:
            # PDF physical pages are zero-indexed in PyMuPDF.
            if page_num < 1 or page_num > len(doc):
                raise ValueError(
                    f"Page {page_num} outside PDF range 1–{len(doc)}"
                )

            page = doc[page_num - 1]
            pix = page.get_pixmap(
                matrix=fitz.Matrix(RENDER_ZOOM, RENDER_ZOOM),
                alpha=False
            )
            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

        safe_name = re.sub(
            r"[^A-Za-z0-9_.-]+", "_", Path(pdf_name).stem
        )

        image_path = PAGE_DIR / (
            f"{idx:03d}_{safe_name}_page_{page_num:04d}.png"
        )

        image.save(image_path, format="PNG")
        rendered_images.append({
            "order": idx,
            "pdf_file": pdf_name,
            "page_number": page_num,
            "image_path": image_path,
            "image": image,
            "review_score": getattr(row, "review_score", None),
            "priority": getattr(row, "review_priority", ""),
        })

        manifest.append({
            "selection_order": idx,
            "pdf_file": pdf_name,
            "page_number": page_num,
            "image_path": str(image_path),
            "status": "RENDERED",
            "review_score": getattr(row, "review_score", None),
            "hydrogeology_relevance": getattr(
                row, "hydrogeology_relevance", None
            ),
        })

    except Exception as exc:
        manifest.append({
            "selection_order": idx,
            "pdf_file": pdf_name,
            "page_number": page_num,
            "image_path": "",
            "status": f"ERROR: {exc}",
            "review_score": getattr(row, "review_score", None),
            "hydrogeology_relevance": getattr(
                row, "hydrogeology_relevance", None
            ),
        })
        print(f"Render error: {pdf_name}, page {page_num}: {exc}")

# -------------------- Create contact sheets --------------------
try:
    font = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18
    )
except Exception:
    font = ImageFont.load_default()

per_sheet = SHEET_COLUMNS * SHEET_ROWS
sheet_paths = []

for start in range(0, len(rendered_images), per_sheet):
    batch = rendered_images[start:start + per_sheet]

    thumb_height = int(
        (batch[0]["image"].height / batch[0]["image"].width)
        * THUMBNAIL_WIDTH
    )

    label_height = 72
    cell_height = thumb_height + label_height
    margin = 20

    sheet_width = (
        SHEET_COLUMNS * THUMBNAIL_WIDTH
        + (SHEET_COLUMNS + 1) * margin
    )
    sheet_height = (
        SHEET_ROWS * cell_height
        + (SHEET_ROWS + 1) * margin
    )

    sheet = Image.new(
        "RGB", (sheet_width, sheet_height), "white"
    )
    draw = ImageDraw.Draw(sheet)

    for local_idx, item in enumerate(batch):
        row_idx = local_idx // SHEET_COLUMNS
        col_idx = local_idx % SHEET_COLUMNS

        x = margin + col_idx * (THUMBNAIL_WIDTH + margin)
        y = margin + row_idx * (cell_height + margin)

        thumb = item["image"].copy()
        thumb.thumbnail(
            (THUMBNAIL_WIDTH, thumb_height),
            Image.Resampling.LANCZOS
        )

        sheet.paste(thumb, (x, y))

        label = (
            f"Review #{item['order']} | PDF page {item['page_number']}\n"
            f"{item['pdf_file']}\n"
            f"Priority: {item['priority']} | "
            f"Score: {item['review_score']}"
        )

        draw.text(
            (x, y + thumb_height + 5),
            label,
            fill="black",
            font=font
        )

    sheet_number = start // per_sheet + 1
    sheet_path = SHEET_DIR / f"contact_sheet_{sheet_number:02d}.jpg"
    sheet.save(sheet_path, quality=92)
    sheet_paths.append(sheet_path)

# -------------------- Save manifest and review register --------------------
manifest_df = pd.DataFrame(manifest)

manifest_file = OUT_DIR / "01_visual_review_manifest.csv"
manifest_df.to_csv(
    manifest_file, index=False, encoding="utf-8-sig"
)

review_file = OUT_DIR / "02_selected_review_pages.csv"

selected.to_csv(
    review_file, index=False, encoding="utf-8-sig"
)

summary_file = OUT_DIR / "03_visual_review_summary.txt"

summary_file.write_text(
    "\n".join([
        "ALWAR BDR VISUAL REVIEW PACKET",
        "=" * 45,
        f"PDF directory: {PDF_ROOT}",
        f"Comparison file: {COMPARISON_FILE}",
        f"Pages selected: {len(selected)}",
        f"Pages rendered: {len(rendered_images)}",
        f"Contact sheets created: {len(sheet_paths)}",
        f"Individual page images: {PAGE_DIR}",
        f"Contact sheets: {SHEET_DIR}",
        "",
        "IMPORTANT:",
        "Images are rendered from original PDFs.",
        "The selected pages are candidates for manual review.",
        "OCR differences are not proof of incorrect measurements.",
        "Verify table columns, units, borehole identity and depth intervals.",
    ]),
    encoding="utf-8"
)

# -------------------- Print results --------------------
print("\n" + "=" * 65)
print("VISUAL REVIEW PACKET COMPLETE")
print("=" * 65)
print("Selected pages:", len(selected))
print("Successfully rendered:", len(rendered_images))
print("Contact sheets:", len(sheet_paths))
print("Output folder:", OUT_DIR)

print("\nContact sheets:")
for p in sheet_paths:
    print(" -", p)

print("\nManifest:", manifest_file)
print("Selected page list:", review_file)
print("Summary:", summary_file)

print("\nReview the contact sheets, then open the individual PNG")
print("for any page where text, depths, or table columns are unclear.")


**[orig #22]** Render selected pages at 300 DPI for manual reading  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import pymupdf

DRIVE = Path("/content/drive/MyDrive")

PDF_ROOT = DRIVE / "gw_framework/data/BDR/Alwar"

OUT = (
    DRIVE / "Alwar_BDR_Lithology_Inventory"
    / "Targeted_Lithology_Verification"
)

OUT.mkdir(parents=True, exist_ok=True)

# Exact pages selected from the uploaded contact sheet.
TARGETS = [
    {
        "pdf": "RWSRP RGWD_BDR_ALWAR 2003 .pdf",
        "page": 284,
        "purpose": "Printed lithological log and pipe assembly",
    },
    {
        "pdf": "RWSRP RGWD_BDR_ALWAR 2003 .pdf",
        "page": 467,
        "purpose": "Printed lithological log; footer indicates Bansur",
    },
    {
        "pdf": "RWSRP RGWD_BDR_ALWAR 2003 .pdf",
        "page": 540,
        "purpose": "Printed lithological log and pipe assembly",
    },
    {
        "pdf": "ALWAR BDR 3 2011 part 2.pdf",
        "page": 6,
        "purpose": "Scanned borehole report and lithological table",
    },
]

# Find PDFs by exact basename.
pdf_lookup = {
    p.name.casefold(): p
    for p in PDF_ROOT.rglob("*.pdf")
}

records = []

for target in TARGETS:
    pdf_name = target["pdf"]
    page_num = target["page"]
    pdf_path = pdf_lookup.get(pdf_name.casefold())

    if pdf_path is None:
        records.append({
            **target,
            "status": "PDF_NOT_FOUND",
            "image_path": "",
        })
        print("PDF not found:", pdf_name)
        continue

    try:
        with pymupdf.open(pdf_path) as doc:
            if page_num < 1 or page_num > len(doc):
                raise ValueError(
                    f"Requested page {page_num}; "
                    f"PDF has {len(doc)} pages"
                )

            page = doc[page_num - 1]

            # Render at approximately 300 DPI.
            pix = page.get_pixmap(
                matrix=pymupdf.Matrix(3.0, 3.0),
                alpha=False
            )

            image_path = OUT / (
                f"{Path(pdf_name).stem.replace(' ', '_')}"
                f"_page_{page_num:04d}.png"
            )

            pix.save(str(image_path))

        records.append({
            **target,
            "status": "RENDERED",
            "source_pdf": str(pdf_path),
            "image_path": str(image_path),
        })

        print(f"Rendered: {pdf_name} | PDF page {page_num}")

    except Exception as exc:
        records.append({
            **target,
            "status": f"ERROR: {exc}",
            "image_path": "",
        })

manifest = pd.DataFrame(records)

manifest_path = OUT / "targeted_page_manifest.csv"
manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nOutput folder:", OUT)
print("Manifest:", manifest_path)
print("\n", manifest.to_string(index=False))


## 3. CGWB hydraulic parameters

Download the CGWB Alwar district report in your browser and upload it to `Alwar_BDR_Lithology_Inventory/CGWB_Automated_Retrieval/downloaded_pdfs`, then rerun cell 28.

**[orig #23]** Hydraulic parameter interpolation from verified CGWB point records (waits for an upload — was interrupted)  
<sub>⏸ interrupted (long run or waiting for upload)</sub>

In [ ]:

# ================================================================
# ALWAR CGWB HYDRAULIC PARAMETER INTERPOLATION
# Source policy: CGWB data only; no fabricated observations.
# Method: IDW interpolation inside the convex hull of data points.
# Outputs: CSV grids, QC reports, and PNG maps in Google Drive.
# ================================================================

from google.colab import drive, files
drive.mount('/content/drive')

import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree, Delaunay
from scipy.interpolate import griddata
from pyproj import Transformer

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- USER SETTINGS ----------------
OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Alwar_BDR_Lithology_Inventory/CGWB_Interpolation"
)

GRID_SPACING_KM = 2.0
IDW_POWER = 2.0
NEIGHBOURS = 8
MIN_POINTS_WARNING = 5

# Set True only if the source coordinates are latitude/longitude.
INPUT_COORDINATES_ARE_LATLON = True

# Expected input column names (case-insensitive; aliases supported).
# Required: latitude, longitude, transmissivity_m2_day
# Optional: aquifer_thickness_m, source_report, source_page, well_id

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- LOAD VERIFIED CGWB DATA ----------------
print("Upload a CSV or Excel file containing verified CGWB point records.")
print("Do not upload a district summary containing only minimum/maximum values.")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Upload the verified CGWB point-data table.")

filename = next(iter(uploaded))
file_bytes = uploaded[filename]

if filename.lower().endswith(".csv"):
    raw = pd.read_csv(io.BytesIO(file_bytes))
elif filename.lower().endswith((".xlsx", ".xls")):
    raw = pd.read_excel(io.BytesIO(file_bytes))
else:
    raise ValueError("Input must be CSV, XLSX, or XLS.")

# ---------------- NORMALIZE COLUMN NAMES ----------------
def norm_col(s):
    return (
        str(s).strip().lower()
        .replace(" ", "_").replace("-", "_")
        .replace("/", "_").replace("(", "")
        .replace(")", "").replace(".", "")
    )

raw.columns = [norm_col(c) for c in raw.columns]

aliases = {
    "latitude": ["latitude", "lat", "latitude_decimal_degrees"],
    "longitude": ["longitude", "lon", "long", "longitude_decimal_degrees"],
    "transmissivity_m2_day": [
        "transmissivity_m2_day", "transmissivity",
        "transmissivity_m2_d", "t_m2_day", "t"
    ],
    "aquifer_thickness_m": [
        "aquifer_thickness_m", "saturated_thickness_m",
        "saturated_aquifer_thickness_m", "thickness_m"
    ],
    "source_report": ["source_report", "report", "document"],
    "source_page": ["source_page", "page", "page_number"],
    "well_id": ["well_id", "well", "borehole_id", "site_id"]
}

rename_map = {}
for target, candidates in aliases.items():
    for candidate in candidates:
        if candidate in raw.columns:
            rename_map[candidate] = target
            break

df = raw.rename(columns=rename_map).copy()

required = ["latitude", "longitude", "transmissivity_m2_day"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        "Rename the source columns to latitude, longitude, "
        "transmissivity_m2_day and rerun."
    )

for col in ["latitude", "longitude", "transmissivity_m2_day",
            "aquifer_thickness_m"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Retain source fields and report rejected records separately.
df["input_row_number"] = np.arange(2, len(df) + 2)
valid = (
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
    & df["transmissivity_m2_day"].gt(0)
)
rejected = df.loc[~valid].copy()
df = df.loc[valid].copy()

if df.empty:
    raise ValueError("No valid positive transmissivity observations remain.")

# Geographic duplicate locations: average only exact coordinate duplicates.
# Preserve a separate table of original records for audit.
df.to_csv(os.path.join(OUTPUT_DIR, "01_valid_source_records.csv"), index=False)
rejected.to_csv(os.path.join(OUTPUT_DIR, "02_rejected_records.csv"), index=False)

# ---------------- PROJECT COORDINATES ----------------
# Alwar is generally in UTM Zone 43N. Use projected metres for IDW.
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
to_geo = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)

df["x_m"], df["y_m"] = to_utm.transform(
    df["longitude"].to_numpy(),
    df["latitude"].to_numpy()
)

# Duplicate coordinates: average T at that location.
# If aquifer thickness is available, average its valid values at that location.
agg = {"transmissivity_m2_day": "mean"}
if "aquifer_thickness_m" in df.columns:
    agg["aquifer_thickness_m"] = "mean"

points = (
    df.groupby(["x_m", "y_m"], as_index=False)
      .agg(agg)
)

n = len(points)
print(f"\nValid source rows: {len(df)}")
print(f"Unique projected locations: {n}")
print(f"Rejected source rows: {len(rejected)}")

if n < 3:
    raise ValueError(
        "At least 3 unique non-collinear locations are needed "
        "for a 2-D interpolation surface."
    )
if n < MIN_POINTS_WARNING:
    print("WARNING: Very few locations; interpolation is highly uncertain.")
if n < 10:
    print("WARNING: Treat the map as exploratory, not a validated regional surface.")

# ---------------- BUILD INTERPOLATION GRID ----------------
xy = points[["x_m", "y_m"]].to_numpy()
T = points["transmissivity_m2_day"].to_numpy()

spacing = GRID_SPACING_KM * 1000
x_grid = np.arange(xy[:, 0].min(), xy[:, 0].max() + spacing, spacing)
y_grid = np.arange(xy[:, 1].min(), xy[:, 1].max() + spacing, spacing)
XX, YY = np.meshgrid(x_grid, y_grid)
grid_xy = np.column_stack([XX.ravel(), YY.ravel()])

# Restrict estimates to the convex hull; do not extrapolate beyond it.
hull = Delaunay(xy)
inside = hull.find_simplex(grid_xy) >= 0

# IDW, exact at source locations. All input T values must be positive.
tree = cKDTree(xy)
k = min(NEIGHBOURS, n)
dist, idx = tree.query(grid_xy, k=k)

if k == 1:
    dist = dist[:, None]
    idx = idx[:, None]

tiny = 1e-9
weights = 1.0 / np.maximum(dist, tiny) ** IDW_POWER
T_grid = np.sum(weights * T[idx], axis=1) / np.sum(weights, axis=1)

# Where a grid cell coincides with a source point, use its exact T.
nearest_dist, nearest_idx = tree.query(grid_xy, k=1)
exact = nearest_dist < 1e-6
T_grid[exact] = T[nearest_idx[exact]]

T_grid[~inside] = np.nan

# ---------------- OPTIONAL CONDUCTIVITY ----------------
# K = T / b, where b is saturated aquifer thickness.
# If thickness is available at point locations, interpolate thickness
# separately, then calculate K. This is an estimate, not a measured K map.
b_grid = np.full(len(grid_xy), np.nan)
K_grid = np.full(len(grid_xy), np.nan)

if "aquifer_thickness_m" in points.columns:
    b = points["aquifer_thickness_m"].to_numpy()
    good_b = np.isfinite(b) & (b > 0)

    if good_b.sum() >= 3:
        b_points = xy[good_b]
        b_values = b[good_b]
        b_tree = cKDTree(b_points)
        db, ib = b_tree.query(grid_xy, k=min(NEIGHBOURS, len(b_values)))

        if db.ndim == 1:
            db, ib = db[:, None], ib[:, None]

        wb = 1.0 / np.maximum(db, tiny) ** IDW_POWER
        b_grid = np.sum(wb * b_values[ib], axis=1) / np.sum(wb, axis=1)

        db_near, ib_near = b_tree.query(grid_xy, k=1)
        b_grid[db_near < 1e-6] = b_values[ib_near[db_near < 1e-6]]

        b_grid[~inside] = np.nan
        valid_k = inside & np.isfinite(T_grid) & np.isfinite(b_grid) & (b_grid > 0)
        K_grid[valid_k] = T_grid[valid_k] / b_grid[valid_k]
        print("Conductivity calculated where positive thickness estimates exist.")
    else:
        print(
            "Thickness column exists, but fewer than 3 valid thickness locations "
            "are available. Conductivity map not produced."
        )
else:
    print(
        "No aquifer thickness column: conductivity is NOT calculated. "
        "Transmissivity map only."
    )

# ---------------- EXPORT GRID ----------------
lon_grid, lat_grid = to_geo.transform(grid_xy[:, 0], grid_xy[:, 1])

grid = pd.DataFrame({
    "longitude": lon_grid,
    "latitude": lat_grid,
    "x_utm43n_m": grid_xy[:, 0],
    "y_utm43n_m": grid_xy[:, 1],
    "inside_source_convex_hull": inside,
    "transmissivity_idw_m2_day": T_grid,
    "interpolated_thickness_m": b_grid,
    "estimated_conductivity_m_day": K_grid
})

grid.to_csv(
    os.path.join(OUTPUT_DIR, "03_interpolated_grid.csv"),
    index=False
)

# ---------------- MAPS ----------------
def save_map(values, title, colorbar_label, filename):
    vals = np.asarray(values).reshape(XX.shape)
    masked = np.ma.masked_invalid(vals)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(XX / 1000, YY / 1000, masked, shading="auto")
    ax.scatter(xy[:, 0] / 1000, xy[:, 1] / 1000,
               marker="o", s=20, edgecolors="black",
               facecolors="none", label="CGWB source locations")
    ax.set_title(title)
    ax.set_xlabel("UTM Zone 43N Easting (km)")
    ax.set_ylabel("UTM Zone 43N Northing (km)")
    ax.legend()
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(colorbar_label)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.show()
    plt.close(fig)

save_map(
    T_grid, "Alwar CGWB Transmissivity — IDW",
    "Transmissivity (m²/day)", "04_transmissivity_idw.png"
)

if np.isfinite(K_grid).any():
    save_map(
        K_grid, "Alwar Estimated Hydraulic Conductivity — IDW",
        "Hydraulic conductivity (m/day)", "05_conductivity_idw.png"
    )

# ---------------- QC SUMMARY ----------------
summary = pd.DataFrame([{
    "input_filename": filename,
    "input_rows": len(raw),
    "valid_source_rows": len(df),
    "unique_locations": n,
    "rejected_rows": len(rejected),
    "transmissivity_min_m2_day": float(T.min()),
    "transmissivity_max_m2_day": float(T.max()),
    "grid_spacing_km": GRID_SPACING_KM,
    "idw_power": IDW_POWER,
    "neighbours": k,
    "grid_cells_inside_hull": int(inside.sum()),
    "conductivity_map_created": bool(np.isfinite(K_grid).any()),
    "method": "IDW; interpolation restricted to convex hull",
    "warning": (
        "Exploratory interpolation. Validate spatial coverage and "
        "withheld-point errors before MODFLOW use."
    )
}])

summary.to_csv(os.path.join(OUTPUT_DIR, "06_QC_summary.csv"), index=False)

print("\nCompleted.")
print("Output folder:", OUTPUT_DIR)
print("\nFiles created:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

print("\nIMPORTANT:")
print("1. Only uploaded CGWB records were used as source observations.")
print("2. District summary minimum/maximum values were not treated as points.")
print("3. IDW estimates between points are model-derived, not observed CGWB values.")
print("4. Do not use this surface as calibrated MODFLOW input without validation.")


**[orig #24]** CGWB Alwar report retrieval + pumping-test extraction  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# CGWB ALWAR REPORT RETRIEVAL + PUMPING-TEST DATA EXTRACTION
# Google Colab | CGWB sources only
#
# Outputs:
#   01_download_log.csv
#   02_pdf_page_text.csv
#   03_hydraulic_evidence.csv
#   04_numeric_evidence_for_review.csv
#   05_candidate_table_extraction.csv
#   06_extraction_summary.txt
#   downloaded_pdfs/
#
# IMPORTANT:
# - Candidate text is not automatically accepted as verified data.
# - No synthetic coordinates or pumping-test values are generated.
# - A district-wide T range is not treated as point observations.
# ================================================================

!pip -q install pymupdf requests beautifulsoup4 pandas openpyxl pdfplumber

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import time
import json
import requests
import pandas as pd
import fitz
import pdfplumber

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# ---------------- CONFIGURATION ----------------

OUT = (
    "/content/drive/MyDrive/"
    "Alwar_BDR_Lithology_Inventory/CGWB_Automated_Retrieval"
)
PDF_DIR = os.path.join(OUT, "downloaded_pdfs")
TEXT_DIR = os.path.join(OUT, "extracted_text")

os.makedirs(OUT, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

# Official CGWB sources only
SOURCES = [
    {
        "name": "Alwar District NAQUIM 2017",
        "landing_url": (
            "https://cgwb.gov.in/cgwbpnm/public/publication-detail/484"
        ),
        "pdf_urls": [
            "https://www.cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
            "https://cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
        ],
    },
    {
        "name": "Neemrana Industrial Area NAQUIM 2.0 2024",
        "landing_url": (
            "https://cgwb.gov.in/cgwbpnm/public/publication-detail/1651"
        ),
        "pdf_urls": [
            "https://cgwb.gov.in/cgwbpnm/public/uploads/documents/"
            "1751320260509508749file.pdf",
        ],
    },
    {
        "name": "Alwar District at a Glance",
        "landing_url": (
            "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf"
        ),
        "pdf_urls": [
            "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
        ],
    },
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 Chrome/124 Safari/537.36"
    )
}
TIMEOUT_SECONDS = 45

# Phrases used to find candidate evidence in report text
HYDRAULIC_TERMS = [
    "transmissivity", "transmissibility", "hydraulic conductivity",
    "coefficient of permeability", "permeability", "storativity",
    "storage coefficient", "specific yield", "pumping test",
    "aquifer test", "drawdown", "discharge", "recovery test",
    "step drawdown", "constant discharge", "specific capacity",
    "observation well", "exploratory well", "hydraulic parameter",
]

# ---------------- HTTP HELPERS ----------------

session = requests.Session()
session.headers.update(HEADERS)

def safe_filename(name):
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name.strip("._")[:150] or "cgwb_report"

def is_pdf_response(response):
    content_type = response.headers.get("Content-Type", "").lower()
    prefix = response.content[:1024].lstrip()
    return (
        "application/pdf" in content_type
        or prefix.startswith(b"%PDF-")
    )

def discover_pdf_links(landing_url):
    """Return PDF URLs exposed by an official CGWB page."""
    links = []
    try:
        response = session.get(
            landing_url, timeout=TIMEOUT_SECONDS
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = urljoin(landing_url, a["href"])
            text = a.get_text(" ", strip=True).lower()
            href_lower = href.lower()

            if (
                ".pdf" in urlparse(href).path.lower()
                or "download" in text
                or "file" in text
            ):
                if urlparse(href).netloc.lower().endswith(
                    ("cgwb.gov.in", "www.cgwb.gov.in")
                ):
                    links.append(href)

        # Some pages expose a direct PDF URL in embedded attributes.
        for tag in soup.find_all(["iframe", "embed", "object"]):
            for attr in ["src", "data"]:
                value = tag.get(attr)
                if value:
                    href = urljoin(landing_url, value)
                    if (
                        ".pdf" in href.lower()
                        and urlparse(href).netloc.lower().endswith(
                            ("cgwb.gov.in", "www.cgwb.gov.in")
                        )
                    ):
                        links.append(href)

    except Exception as e:
        print(f"Landing page unavailable: {landing_url} | {e}")

    return list(dict.fromkeys(links))

def download_pdf(url, target_path):
    """Download and validate a PDF; return status and details."""
    try:
        r = session.get(url, timeout=TIMEOUT_SECONDS)
        r.raise_for_status()

        if not is_pdf_response(r):
            return False, (
                f"URL returned non-PDF content "
                f"(Content-Type={r.headers.get('Content-Type')})"
            )

        if len(r.content) < 1000:
            return False, f"Response suspiciously small ({len(r.content)} bytes)"

        with open(target_path, "wb") as f:
            f.write(r.content)

        # Verify the saved file can be opened as a PDF.
        with fitz.open(target_path) as doc:
            page_count = len(doc)

        return True, (
            f"Downloaded {len(r.content)} bytes; "
            f"{page_count} pages"
        )

    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

# ---------------- RETRIEVE REPORTS ----------------

download_log = []
downloaded = []

for source in SOURCES:
    name = source["name"]
    candidates = list(source["pdf_urls"])
    candidates += discover_pdf_links(source["landing_url"])
    candidates = list(dict.fromkeys(candidates))

    safe = safe_filename(name)
    target = os.path.join(PDF_DIR, safe + ".pdf")
    success = False

    # If a previous valid download exists, reuse it.
    if os.path.exists(target):
        try:
            with fitz.open(target) as doc:
                if len(doc) > 0:
                    success = True
                    downloaded.append({
                        "source_name": name,
                        "url": "Previously downloaded local copy",
                        "path": target,
                    })
                    download_log.append({
                        "source_name": name,
                        "url": "Previously downloaded local copy",
                        "status": "REUSED",
                        "details": f"{len(doc)} pages",
                    })
        except Exception:
            success = False

    if success:
        continue

    for url in candidates:
        # Keep retrieval within official CGWB domains.
        host = urlparse(url).netloc.lower()
        if not host.endswith(("cgwb.gov.in", "www.cgwb.gov.in")):
            continue

        ok, details = download_pdf(url, target)

        download_log.append({
            "source_name": name,
            "url": url,
            "status": "SUCCESS" if ok else "FAILED",
            "details": details,
        })

        if ok:
            downloaded.append({
                "source_name": name,
                "url": url,
                "path": target,
            })
            print(f"SUCCESS: {name} -> {target}")
            success = True
            break
        else:
            print(f"FAILED: {name} | {url} | {details}")

        time.sleep(1)

    if not success:
        download_log.append({
            "source_name": name,
            "url": source["landing_url"],
            "status": "NO_PDF_DOWNLOADED",
            "details": "All discovered official PDF URLs failed.",
        })

pd.DataFrame(download_log).to_csv(
    os.path.join(OUT, "01_download_log.csv"), index=False
)

print("\nSuccessfully available PDFs:", len(downloaded))

# ---------------- EXTRACT ALL PAGE TEXT ----------------

page_records = []
evidence_records = []
numeric_records = []

# A conservative numeric pattern for review only.
number_pattern = re.compile(
    r"(?<![A-Za-z])[-+]?\d+(?:,\d{3})*(?:\.\d+)?"
)

for item in downloaded:
    pdf_path = item["path"]
    source_name = item["source_name"]
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    text_output = os.path.join(TEXT_DIR, base + "_page_text.txt")

    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print("Could not open PDF:", pdf_path, e)
        continue

    with open(text_output, "w", encoding="utf-8") as text_file:
        for page_index, page in enumerate(doc):
            page_num = page_index + 1
            text = page.get_text("text", sort=True) or ""
            text_file.write(
                f"\n\n===== PDF PAGE {page_num} =====\n{text}"
            )

            page_records.append({
                "source_name": source_name,
                "pdf_filename": os.path.basename(pdf_path),
                "pdf_page": page_num,
                "page_text_chars": len(text),
                "has_text": bool(text.strip()),
                "page_text": text,
            })

            lower = text.lower()
            matched_terms = [
                term for term in HYDRAULIC_TERMS if term in lower
            ]

            if matched_terms:
                # Split page into lines and keep relevant context.
                lines = text.splitlines()
                for i, line in enumerate(lines):
                    line_lower = line.lower()
                    found = [
                        term for term in HYDRAULIC_TERMS
                        if term in line_lower
                    ]
                    if found:
                        start = max(0, i - 2)
                        end = min(len(lines), i + 3)
                        context = " | ".join(
                            x.strip() for x in lines[start:end]
                            if x.strip()
                        )

                        evidence_records.append({
                            "source_name": source_name,
                            "pdf_filename": os.path.basename(pdf_path),
                            "pdf_page": page_num,
                            "matched_terms": "; ".join(found),
                            "evidence_line": line.strip(),
                            "context": context,
                            "verification_status": "CANDIDATE_NOT_VERIFIED",
                        })

            # Keep numeric evidence only on relevant pages.
            if matched_terms:
                nums = number_pattern.findall(text)
                numeric_records.append({
                    "source_name": source_name,
                    "pdf_filename": os.path.basename(pdf_path),
                    "pdf_page": page_num,
                    "matched_terms": "; ".join(matched_terms),
                    "numeric_tokens": "; ".join(nums),
                    "page_text": text,
                    "verification_status": "MANUAL_TABLE_REVIEW_REQUIRED",
                })

    doc.close()
    print("Extracted text:", source_name)

pages_df = pd.DataFrame(page_records)
evidence_df = pd.DataFrame(evidence_records)
numeric_df = pd.DataFrame(numeric_records)

pages_df.to_csv(
    os.path.join(OUT, "02_pdf_page_text.csv"), index=False
)
evidence_df.to_csv(
    os.path.join(OUT, "03_hydraulic_evidence.csv"), index=False
)
numeric_df.to_csv(
    os.path.join(OUT, "04_numeric_evidence_for_review.csv"), index=False
)

# ---------------- TABLE EXTRACTION CANDIDATES ----------------
# Extract page tables using pdfplumber. Layout and merged cells may be
# imperfect; every output table remains a candidate for visual checking.

table_records = []

for item in downloaded:
    pdf_path = item["path"]
    source_name = item["source_name"]

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_index, page in enumerate(pdf.pages):
                page_text = page.extract_text() or ""
                lower = page_text.lower()

                if not any(term in lower for term in HYDRAULIC_TERMS):
                    continue

                try:
                    tables = page.extract_tables()
                except Exception:
                    tables = []

                for table_idx, table in enumerate(tables or [], start=1):
                    if not table:
                        continue

                    for row_idx, row in enumerate(table, start=1):
                        cleaned = [
                            "" if value is None else str(value).strip()
                            for value in row
                        ]
                        table_records.append({
                            "source_name": source_name,
                            "pdf_filename": os.path.basename(pdf_path),
                            "pdf_page": page_index + 1,
                            "table_number_on_page": table_idx,
                            "row_number_in_table": row_idx,
                            "row_cells": json.dumps(
                                cleaned, ensure_ascii=False
                            ),
                            "verification_status": "CANDIDATE_NOT_VERIFIED",
                        })

    except Exception as e:
        print("Table extraction issue:", pdf_path, e)

tables_df = pd.DataFrame(table_records)
tables_df.to_csv(
    os.path.join(OUT, "05_candidate_table_extraction.csv"),
    index=False
)

# ---------------- SUMMARY ----------------

summary_lines = [
    "CGWB ALWAR REPORT RETRIEVAL SUMMARY",
    "===================================",
    f"Reports successfully available: {len(downloaded)}",
    f"PDF pages with records: {len(pages_df)}",
    f"Pages with extractable text: "
    f"{int(pages_df['has_text'].sum()) if len(pages_df) else 0}",
    f"Hydraulic evidence contexts: {len(evidence_df)}",
    f"Numeric candidate pages: {len(numeric_df)}",
    f"Candidate extracted table rows: {len(tables_df)}",
    "",
    "IMPORTANT INTERPRETATION RULES",
    "- Keyword matches are candidates, not verified hydraulic parameters.",
    "- Numeric tokens are not automatically assigned to columns or units.",
    "- No missing well coordinates have been invented.",
    "- No district-level range was converted to synthetic point data.",
    "- Check PDF page images before accepting table values.",
    "- Calculate K=T/b only where CGWB supports both T and thickness.",
    "",
    f"Output directory: {OUT}",
]

summary_path = os.path.join(OUT, "06_extraction_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))
print("\nOutput files:")
for f in sorted(os.listdir(OUT)):
    print(" -", f)

# ---------------- OPTIONAL: ZIP DOWNLOADED REPORTS ----------------
# Creates a ZIP of successfully downloaded PDFs and extracted outputs.
import zipfile

zip_path = os.path.join(OUT, "CGWB_Alwar_Retrieval_Outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files_list in os.walk(OUT):
        # Do not include the ZIP inside itself.
        for file in files_list:
            full = os.path.join(root, file)
            if full == zip_path:
                continue
            z.write(full, arcname=os.path.relpath(full, OUT))

print("\nZIP created:", zip_path)
print("If a CGWB download failed, inspect 01_download_log.csv.")


**[orig #25]** CGWB download retry — SSL diagnostics  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# CGWB DOWNLOAD RETRY — SSL/TLS DIAGNOSTICS
# Run this cell before the PDF extraction section.
# ================================================================

!pip -q install --upgrade certifi requests pymupdf

import os
import ssl
import certifi
import requests
import pandas as pd
import pymupdf
from urllib.parse import urlparse

OUT = (
    "/content/drive/MyDrive/"
    "Alwar_BDR_Lithology_Inventory/CGWB_Automated_Retrieval"
)
PDF_DIR = os.path.join(OUT, "downloaded_pdfs")
os.makedirs(PDF_DIR, exist_ok=True)

# Official CGWB URLs only; alternate hostnames are tested separately.
SOURCES = [
    {
        "name": "Alwar_NAQUIM_2017",
        "urls": [
            "https://cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
            "https://www.cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
            "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
            "https://www.cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
        ],
    },
    {
        "name": "Neemrana_NAQUIM_2_0",
        "urls": [
            "https://cgwb.gov.in/cgwbpnm/public/uploads/documents/"
            "1751320260509508749file.pdf",
            "https://www.cgwb.gov.in/cgwbpnm/public/uploads/documents/"
            "1751320260509508749file.pdf",
        ],
    },
]

# Use an updated, trusted certificate bundle.
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) "
                  "AppleWebKit/537.36 Chrome/125 Safari/537.36",
    "Accept": "application/pdf,*/*;q=0.8",
})
session.verify = certifi.where()

print("Python SSL:", ssl.OPENSSL_VERSION)
print("Certificate bundle:", certifi.where())

downloaded = []
log = []

for source in SOURCES:
    name = source["name"]
    output_path = os.path.join(PDF_DIR, name + ".pdf")
    success = False

    for url in source["urls"]:
        host = urlparse(url).hostname or ""

        if not (host == "cgwb.gov.in" or host == "www.cgwb.gov.in"):
            log.append({
                "source": name, "url": url,
                "status": "BLOCKED_NON_CGWB_HOST",
                "details": host
            })
            continue

        try:
            print(f"\nTrying: {url}")
            response = session.get(
                url, timeout=(15, 90), allow_redirects=True
            )
            response.raise_for_status()

            # Do not accept HTML error pages saved with a .pdf extension.
            if not response.content.startswith(b"%PDF-"):
                raise ValueError(
                    "Response is not a PDF; content-type="
                    + response.headers.get("Content-Type", "unknown")
                )

            with open(output_path, "wb") as f:
                f.write(response.content)

            # Verify that the saved PDF opens and has pages.
            with pymupdf.open(output_path) as doc:
                page_count = len(doc)

            if page_count == 0:
                raise ValueError("PDF contains no pages.")

            downloaded.append({
                "source_name": name,
                "url": response.url,
                "path": output_path,
            })

            log.append({
                "source": name, "url": url,
                "status": "SUCCESS",
                "details": (
                    f"{len(response.content)} bytes; "
                    f"{page_count} pages; final URL={response.url}"
                )
            })
            print(f"SUCCESS: {page_count} pages downloaded.")
            success = True
            break

        except Exception as e:
            log.append({
                "source": name,
                "url": url,
                "status": "FAILED",
                "details": f"{type(e).__name__}: {e}"
            })
            print(f"FAILED: {type(e).__name__}: {e}")

    if not success:
        print(f"\nCould not download {name} automatically.")

log_path = os.path.join(OUT, "01_download_retry_log.csv")
pd.DataFrame(log).to_csv(log_path, index=False)

print("\n========== DOWNLOAD RESULTS ==========")
print("Successful downloads:", len(downloaded))
print("Log saved:", log_path)

for item in downloaded:
    print(item["source_name"], "->", item["path"])

if not downloaded:
    print(
        "\nCGWB downloads remain unavailable from this Colab session. "
        "Inspect the retry log for the exact error. Do not run extraction "
        "until at least one PDF has downloaded."
    )


**[orig #26]** Download CGWB PDFs with curl and validate  
<sub>✅ ran OK</sub>

In [ ]:

import subprocess
import os

out_dir = (
    "/content/drive/MyDrive/"
    "Alwar_BDR_Lithology_Inventory/CGWB_Automated_Retrieval/downloaded_pdfs"
)
os.makedirs(out_dir, exist_ok=True)

urls = {
    "Alwar_NAQUIM_2017.pdf":
        "https://www.cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
    "Alwar_District_at_a_Glance.pdf":
        "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
    "Neemrana_NAQUIM_2024.pdf":
        "https://cgwb.gov.in/cgwbpnm/public/uploads/documents/"
        "1751320260509508749file.pdf",
}

for filename, url in urls.items():
    destination = os.path.join(out_dir, filename)
    result = subprocess.run(
        [
            "curl", "--fail", "--location",
            "--connect-timeout", "20",
            "--max-time", "180",
            "--output", destination,
            "--write-out",
            "\nHTTP_STATUS=%{http_code}\nREMOTE_IP=%{remote_ip}\n",
            url,
        ],
        capture_output=True,
        text=True,
    )

    print("\n", filename)
    print(result.stdout)
    if result.stderr:
        print(result.stderr[-1500:])

    if os.path.exists(destination):
        with open(destination, "rb") as f:
            is_pdf = f.read(5) == b"%PDF-"

        if is_pdf:
            print("VALID PDF:", destination)
        else:
            print("Not a valid PDF; removing incomplete response.")
            os.remove(destination)


**[orig #27]** DNS-over-HTTPS check for cgwb.gov.in  
<sub>✅ ran OK</sub>

In [ ]:

import requests
import subprocess
import os
import json

domain = "cgwb.gov.in"

# Query Google Public DNS over HTTPS.
try:
    r = requests.get(
        "https://dns.google/resolve",
        params={"name": domain, "type": "A"},
        timeout=20
    )
    r.raise_for_status()
    dns_result = r.json()

    ips = [
        answer["data"]
        for answer in dns_result.get("Answer", [])
        if answer.get("type") == 1
    ]

    print("DNS status:", dns_result.get("Status"))
    print("Google DNS A records:", ips)

except Exception as e:
    ips = []
    print("Google DNS lookup failed:", repr(e))

# Test each returned address while retaining the domain as the
# TLS hostname. curl still verifies the server certificate.
test_url = "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf"

for ip in ips:
    print("\nTesting CGWB with resolved IP:", ip)

    result = subprocess.run(
        [
            "curl", "--fail", "--location",
            "--connect-timeout", "15",
            "--max-time", "60",
            "--resolve", f"cgwb.gov.in:443:{ip}",
            "--output", "/tmp/cgwb_test.pdf",
            "--write-out",
            "HTTP=%{http_code} IP=%{remote_ip}\n",
            test_url
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr[-1000:] if result.stderr else "")

    if os.path.exists("/tmp/cgwb_test.pdf"):
        with open("/tmp/cgwb_test.pdf", "rb") as f:
            valid_pdf = f.read(5) == b"%PDF-"

        if valid_pdf:
            print("Verified PDF downloaded from:", ip)
            break
        else:
            os.remove("/tmp/cgwb_test.pdf")


**[orig #28]** Download + extract hydraulic data (failed: SSL certificate error — download the PDF in a browser instead)  
<sub>✅ ran OK</sub>

In [ ]:

# ================================================================
# CGWB ALWAR: DOWNLOAD + VERIFY + EXTRACT HYDRAULIC DATA
# No package installation. SSL verification remains enabled.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, time, json, socket, subprocess
import requests
import pandas as pd
import pymupdf
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------- PATHS ----------------
BASE = "/content/drive/MyDrive/Alwar_BDR_Lithology_Inventory/CGWB_Automated_Retrieval"
PDF_DIR = os.path.join(BASE, "downloaded_pdfs")
TEXT_DIR = os.path.join(BASE, "extracted_text")
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)

# Official CGWB source URLs
SOURCES = [
    {
        "name": "Alwar_NAQUIM_2017",
        "urls": [
            "https://cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
            "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
            "https://www.cgwb.gov.in/sites/default/files/2022-11/alwar.pdf",
        ],
        "pages": [
            "https://cgwb.gov.in/cgwbpnm/public/publication-detail/484"
        ],
    },
    {
        "name": "Neemrana_NAQUIM_2_0",
        "urls": [
            "https://cgwb.gov.in/cgwbpnm/public/uploads/documents/"
            "1751320260509508749file.pdf",
        ],
        "pages": [
            "https://cgwb.gov.in/cgwbpnm/public/publication-detail/1651"
        ],
    },
    {
        "name": "Alwar_District_at_a_Glance",
        "urls": [
            "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf",
        ],
        "pages": [],
    },
]

TERMS = [
    "transmissivity", "transmissibility", "hydraulic conductivity",
    "coefficient of permeability", "permeability", "storativity",
    "storage coefficient", "specific yield", "pumping test",
    "aquifer test", "drawdown", "discharge", "recovery test",
    "step drawdown", "specific capacity", "observation well",
    "exploratory well", "hydraulic parameter", "water level"
]

# ---------------- SESSION ----------------
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) "
                  "AppleWebKit/537.36 Chrome/126 Safari/537.36",
    "Accept": "application/pdf,text/html,*/*;q=0.8"
})

# Retry only transient HTTP errors. Never disable certificate verification.
retry = Retry(
    total=2, connect=2, read=2, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.verify = True

download_log = []
available_pdfs = []

# ---------------- NETWORK DIAGNOSTICS ----------------
print("Requests version:", requests.__version__)
print("CGWB DNS lookup:")
for host in ["cgwb.gov.in", "www.cgwb.gov.in"]:
    try:
        print(host, "->", socket.getaddrinfo(host, 443)[0][4][0])
    except Exception as e:
        print(host, "DNS error:", repr(e))

# ---------------- PDF VALIDATION ----------------
def is_valid_pdf(path):
    try:
        with open(path, "rb") as f:
            if f.read(5) != b"%PDF-":
                return False, "File does not begin with PDF signature"
        with pymupdf.open(path) as doc:
            n = len(doc)
        if n < 1:
            return False, "PDF has no pages"
        return True, f"Valid PDF, {n} pages"
    except Exception as e:
        return False, f"PDF validation failed: {e}"

def discover_pdf_links(page_url):
    """Discover PDF links from an official CGWB publication page."""
    found = []
    try:
        r = session.get(page_url, timeout=(15, 40))
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for a in soup.find_all("a", href=True):
            u = urljoin(page_url, a["href"])
            host = urlparse(u).hostname or ""
            if (
                host in ["cgwb.gov.in", "www.cgwb.gov.in"]
                and ".pdf" in urlparse(u).path.lower()
            ):
                found.append(u)
    except Exception as e:
        print("Publication-page discovery failed:", page_url, repr(e))
    return list(dict.fromkeys(found))

def try_download(name, urls):
    path = os.path.join(PDF_DIR, name + ".pdf")

    # Reuse existing file only if it passes validation.
    if os.path.exists(path):
        ok, msg = is_valid_pdf(path)
        if ok:
            print("REUSING:", name, msg)
            return path, "REUSED", msg
        else:
            os.remove(path)

    errors = []
    for url in list(dict.fromkeys(urls)):
        host = urlparse(url).hostname or ""
        if host not in ["cgwb.gov.in", "www.cgwb.gov.in"]:
            errors.append(f"Blocked non-CGWB URL: {url}")
            continue

        print("\nTrying:", url)
        try:
            r = session.get(
                url, timeout=(20, 90), allow_redirects=True
            )
            r.raise_for_status()

            if not r.content.startswith(b"%PDF-"):
                raise ValueError(
                    "Server did not return a PDF; "
                    f"Content-Type={r.headers.get('Content-Type')}; "
                    f"final URL={r.url}"
                )

            tmp = path + ".part"
            with open(tmp, "wb") as f:
                f.write(r.content)
            os.replace(tmp, path)

            ok, msg = is_valid_pdf(path)
            if not ok:
                os.remove(path)
                raise ValueError(msg)

            print("SUCCESS:", msg)
            return path, "SUCCESS", msg

        except Exception as e:
            message = f"{type(e).__name__}: {e}"
            errors.append(f"{url} -> {message}")
            print("FAILED:", message)
        time.sleep(1)

    return None, "FAILED", " || ".join(errors)

# ---------------- DOWNLOAD REPORTS ----------------
for source in SOURCES:
    urls = list(source["urls"])
    for page in source["pages"]:
        urls += discover_pdf_links(page)

    path, status, details = try_download(source["name"], urls)

    download_log.append({
        "source_name": source["name"],
        "status": status,
        "details": details,
        "local_path": path or ""
    })

    if path:
        available_pdfs.append({
            "source_name": source["name"],
            "path": path
        })

# Include any other PDFs already manually placed in download folder.
known = {os.path.abspath(x["path"]) for x in available_pdfs}
for filename in os.listdir(PDF_DIR):
    if filename.lower().endswith(".pdf"):
        path = os.path.abspath(os.path.join(PDF_DIR, filename))
        if path not in known:
            ok, msg = is_valid_pdf(path)
            if ok:
                available_pdfs.append({
                    "source_name": os.path.splitext(filename)[0],
                    "path": path
                })
                known.add(path)

pd.DataFrame(download_log).to_csv(
    os.path.join(BASE, "01_download_log.csv"), index=False
)

print("\nPDFs available for extraction:", len(available_pdfs))

# ---------------- STOP SAFELY IF NO PDF ----------------
if not available_pdfs:
    print("""
NO PDF WAS DOWNLOADED OR FOUND LOCALLY.

The script will not create empty extraction outputs.
If your browser can open the report but Colab cannot:
1. Download the PDF in your browser.
2. Upload it into this Drive folder:
   """ + PDF_DIR + """
3. Rerun this same cell.

The script will detect and process the uploaded PDF.
""")
else:
    # ---------------- EXTRACT PAGE TEXT ----------------
    page_rows = []
    evidence_rows = []
    numeric_rows = []
    table_rows = []

    number_pattern = re.compile(
        r"(?<![A-Za-z])[-+]?\d+(?:,\d{3})*(?:\.\d+)?"
    )

    for item in available_pdfs:
        name = item["source_name"]
        path = item["path"]
        print("\nExtracting:", name)

        try:
            doc = pymupdf.open(path)
        except Exception as e:
            print("Cannot open:", path, repr(e))
            continue

        full_text_path = os.path.join(
            TEXT_DIR,
            re.sub(r"[^A-Za-z0-9_.-]", "_", name) + "_text.txt"
        )

        with open(full_text_path, "w", encoding="utf-8") as tf:
            for i, page in enumerate(doc):
                page_no = i + 1
                text = page.get_text("text", sort=True) or ""
                tf.write(f"\n\n===== PDF PAGE {page_no} =====\n{text}")

                page_rows.append({
                    "source_name": name,
                    "pdf_filename": os.path.basename(path),
                    "pdf_page": page_no,
                    "text_chars": len(text),
                    "has_text": bool(text.strip()),
                    "page_text": text
                })

                lower = text.lower()
                matched = [t for t in TERMS if t in lower]
                if not matched:
                    continue

                lines = text.splitlines()
                for j, line in enumerate(lines):
                    line_terms = [t for t in TERMS if t in line.lower()]
                    if line_terms:
                        lo, hi = max(0, j - 3), min(len(lines), j + 4)
                        context = " | ".join(
                            z.strip() for z in lines[lo:hi] if z.strip()
                        )
                        evidence_rows.append({
                            "source_name": name,
                            "pdf_filename": os.path.basename(path),
                            "pdf_page": page_no,
                            "terms": "; ".join(line_terms),
                            "evidence_line": line.strip(),
                            "context": context,
                            "verification": "CANDIDATE_REQUIRES_PAGE_CHECK"
                        })

                numeric_rows.append({
                    "source_name": name,
                    "pdf_filename": os.path.basename(path),
                    "pdf_page": page_no,
                    "terms": "; ".join(matched),
                    "numeric_tokens": "; ".join(number_pattern.findall(text)),
                    "page_text": text,
                    "verification": "NOT_STRUCTURED_OR_VERIFIED"
                })

        doc.close()

    pd.DataFrame(page_rows).to_csv(
        os.path.join(BASE, "02_pdf_page_text.csv"), index=False
    )
    pd.DataFrame(evidence_rows).to_csv(
        os.path.join(BASE, "03_hydraulic_evidence.csv"), index=False
    )
    pd.DataFrame(numeric_rows).to_csv(
        os.path.join(BASE, "04_numeric_evidence_for_review.csv"), index=False
    )

    # ---------------- TABLE CANDIDATES ----------------
    try:
        import pdfplumber

        for item in available_pdfs:
            path, name = item["path"], item["source_name"]
            try:
                with pdfplumber.open(path) as pdf:
                    for i, page in enumerate(pdf.pages):
                        text = page.extract_text() or ""
                        if not any(t in text.lower() for t in TERMS):
                            continue

                        try:
                            tables = page.extract_tables() or []
                        except Exception:
                            tables = []

                        for table_no, table in enumerate(tables, start=1):
                            for row_no, row in enumerate(table, start=1):
                                table_rows.append({
                                    "source_name": name,
                                    "pdf_filename": os.path.basename(path),
                                    "pdf_page": i + 1,
                                    "table_number": table_no,
                                    "row_number": row_no,
                                    "row_cells_json": json.dumps(
                                        row, ensure_ascii=False
                                    ),
                                    "verification": "CANDIDATE_NOT_VERIFIED"
                                })
            except Exception as e:
                print("Table extraction issue:", name, repr(e))

    except ImportError:
        print("pdfplumber unavailable; skipping candidate table extraction.")

    pd.DataFrame(table_rows).to_csv(
        os.path.join(BASE, "05_candidate_table_extraction.csv"),
        index=False
    )

    # ---------------- SUMMARY ----------------
    summary = [
        "CGWB ALWAR AUTOMATED EXTRACTION SUMMARY",
        "=======================================",
        f"PDFs available: {len(available_pdfs)}",
        f"Pages extracted: {len(page_rows)}",
        f"Pages containing text: {sum(x['has_text'] for x in page_rows)}",
        f"Hydraulic evidence lines: {len(evidence_rows)}",
        f"Numeric candidate pages: {len(numeric_rows)}",
        f"Candidate table rows: {len(table_rows)}",
        "",
        "All keyword and numeric matches are candidates, not verified data.",
        "Inspect source pages before assigning values to parameter columns.",
        "Do not treat district-wide T ranges as individual point observations.",
        "Calculate K=T/b only where transmissivity and saturated thickness",
        "are supported by source records.",
        "",
        "Output folder: " + BASE
    ]

    with open(
        os.path.join(BASE, "06_extraction_summary.txt"),
        "w", encoding="utf-8"
    ) as f:
        f.write("\n".join(summary))

    print("\n".join(summary))

# ---------------- ZIP OUTPUTS ----------------
import zipfile

zip_path = os.path.join(BASE, "CGWB_Alwar_Extraction_Outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, filenames in os.walk(BASE):
        for filename in filenames:
            full = os.path.join(root, filename)
            if full == zip_path or filename.endswith(".part"):
                continue
            z.write(full, os.path.relpath(full, BASE))

print("\nZIP:", zip_path)
print("All outputs saved under:", BASE)
